# Script Analysis Pipeline (대본 분석 파이프라인)

슬라이드 대본이 **등록·수정될 때 한 번** 실행되어, 나중에 발표자의 실제 발화(STT)와 비교할 때 쓸
**Evaluation Rubric(평가 기준)** 을 만들고 DB 에 저장합니다. 발표할 때마다 대본을 다시 분석하지 않도록,
대본에서 뽑을 수 있는 것은 여기서 미리 뽑아 둡니다. 아래 그림의 괄호는 이 노트북의 장 번호입니다.

```text
                     Slide Script (슬라이드 대본)
                               │
                       Normalize Script (2장)
                               │
               ┌───────────────┴───────────────┐
               ▼                               ▼
   규칙 기반 분석 1회 (3장)            LLM API 1차 분석 1회 (4장)
   · Critical Fact Parser              · 문장 역할 → Core Claim → Key Point
     숫자·비율·날짜·금액·고유명사
   · TF-IDF / Keyword
               └───────────────┬───────────────┘
                               ▼
                1차 결과 정리·검증 (5장, 코드)
                               ▼
              LLM API 최종 결론 1회 (6장)
              · 1차 분석 + 규칙 분석 + 검증 경고 → 최종 판단
                               ▼
            Evaluation Rubric 조립 (코드) → DB (7장)
```

**구조** — 슬라이드마다 규칙 기반 분석 1회, LLM 1차 분석 1회, LLM 최종 결론 1회입니다.
- 숫자·날짜·이름처럼 규칙으로 확실하게 뽑을 수 있는 것은 코드가, 어떤 내용이 핵심인지처럼 의미를 이해해야 하는 것은 LLM 이 맡습니다.
  두 갈래는 서로의 결과를 기다리지 않고 병렬로 돕니다.
- 1차 결과를 코드가 정리·검증하고, **최종 결론 LLM 이 대본·1차 분석·규칙 분석·검증 경고를 한꺼번에 보고** 문제를 고쳐 최종 판단을 내립니다.
- 최종 판단을 평가 기준으로 조립하는 일(중요도 계산, 사실 연결, 검증)은 다시 코드가 합니다. 수치 사실은 규칙 결과를 그대로 쓰므로 LLM 이 빼지 못합니다.

| 그림 요소 | 함수 |
|---|---|
| (입력) 대본 JSON 읽기 | `load_script_json` |
| Normalize Script | `normalize_script` |
| Critical Fact Parser — 숫자·비율·날짜 / 고유명사 | `extract_critical_facts` = `extract_latin_names` → `extract_numeric_facts`(+`parse_korean_number`) → `extract_korean_proper_nouns` |
| TF-IDF / Keyword | `extract_keywords_tfidf` |
| LLM 1차 분석 — Key Point · Importance · Core Claim | `analyze_semantics` (문장 역할 → Core Claim → Key Point 순서로 받음) |
| 1차 결과 정리·검증 | `draft_rubric` = `resolve_roles` → `make_key_points`(+`importance_from_roles`) → `merge_key_terms` → `link_facts_to_key_points` → `validate_rubric` |
| LLM 최종 결론 | `name_candidates` → `final_user_message` → `review_final` |
| Evaluation Rubric 조립 | `finalize_rubric` (1차와 같은 규칙으로 중요도·사실 연결·검증을 다시 계산) |
| DB | `connect_db` · `save_rubric` · `load_rubric` (+ LLM 캐시 `load/save_cached_semantics` · `load/save_cached_final`) |
| 파이프라인 전체 | `run_script_analysis(JSON 파일 경로)` |
| (점검) LLM 일관성 | `sample_rubrics` · `compare_views` · `score_spread` |

**데이터** — `data/scripts/가상대본1.json` (도서관 좌석 혼잡도 예측 서비스, 9장), `data/scripts/가상대본2.json` (동네 빵집 재고 예측 서비스, 11장).
서비스·팀·사람·수치 모두 지어낸 가상 발표 대본입니다. JSON 파일 하나가 발표 하나이고, 결과는 **파일 이름**(`가상대본1`)으로 저장·조회합니다.

**LLM API** — `.env` 의 `OPENAI_BASE_URL` · `OPENAI_API_KEY` · `OPENAI_MODEL` 로 호출합니다 (슬라이드 1장당 1차 분석 1회 + 최종 결론 1회).
응답은 `outputs/rubrics.sqlite` 에 캐시되어, 같은 입력·같은 프롬프트면 노트북을 다시 실행해도 다시 호출하지 않습니다.
캐시가 빈 상태에서 처음 실행하면 20장 × 2회 + 수정본 2회 + 일관성 측정 80회, 모두 122회 호출합니다 (일관성 측정을 끄려면 12장의 `N_SAMPLES = 1`).

**셋업** — 이 디렉터리 `v1/local/` 에서 노트북을 엽니다. `.env` 는 현재 디렉터리부터 **상위로 올라가며 처음 만나는 파일**을 읽습니다
(키는 `ai/.env` 에 둡니다. 어느 파일을 읽었는지 첫 셀이 출력합니다).

```bash
python -m venv .venv
./.venv/Scripts/python.exe -m pip install -r requirements.txt
```

## 0. 환경 설정

In [1]:
# library import
import hashlib
import json
import os
import random
import re
import sqlite3
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer

# 환경 설정: 현재 디렉터리부터 상위로 올라가며 처음 만나는 .env 를 읽는다 (v1/local/.env → … → ai/.env)
ENV_PATH = find_dotenv(usecwd=True)
load_dotenv(ENV_PATH)
BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL")
assert API_KEY and MODEL, f".env 에 OPENAI_API_KEY / OPENAI_MODEL 이 없다 (읽은 파일: {ENV_PATH or '없음'})"

ROOT = Path.cwd()
SCRIPT_DIR = ROOT / "data" / "scripts"
DB_PATH = ROOT / "outputs" / "rubrics.sqlite"  # LLM 응답 캐시 + 평가 기준

RUBRIC_SCHEMA_VERSION = "1.0"

kiwi = Kiwi()

print(f"MODEL={MODEL}, .env={ENV_PATH}")

MODEL=openai/gpt-5.6-luna, .env=E:\jewon\JWK project\ktc4-pusan-2\ai\.env


## 1. 가상 데이터 준비

대본은 슬라이드 번호와 대본 내용으로 묶인 JSON 입니다.

```json
[
  {"slide_number": 1, "script": "안녕하십니까. 도서관 좌석 혼잡도를 예측해 ..."},
  {"slide_number": 2, "script": "지금 도서관들은 좌석 예약 키오스크, ..."}
]
```

파이프라인 입력은 **이 JSON 파일 하나**입니다. 따로 입력할 값은 없고, 파일 이름이 결과를 저장·조회하는 이름이 됩니다.
대본을 고치면 고친 JSON 을 다른 이름으로 넣으면 됩니다 (10장).

In [2]:
class SlideScript(BaseModel):
    """입력 JSON 의 한 항목: 슬라이드 한 장의 대본."""
    slide_number: int = Field(description="슬라이드 번호")
    script: str = Field(description="슬라이드 대본 원문")


def load_script_json(path: Path) -> list[SlideScript]:
    """대본 JSON 을 읽는다. 형식: [{"slide_number": 1, "script": "..."}, ...]"""
    items = json.loads(path.read_text(encoding="utf-8"))
    return sorted((SlideScript.model_validate(item) for item in items), key=lambda s: s.slide_number)


# 가상 데이터: data/scripts 의 JSON 파일 하나가 발표 하나다. 파일 이름으로 결과를 저장·조회한다
SCRIPT_FILES = sorted(SCRIPT_DIR.glob("*.json"))
slide_scripts = {path.stem: load_script_json(path) for path in SCRIPT_FILES}

pd.DataFrame([
    {"file": name, "slide": s.slide_number, "chars": len(s.script), "script": s.script[:40] + "…"}
    for name, slides in slide_scripts.items() for s in slides
])

,file,slide,chars,script
0,가상대본1,1,264,안녕하십니까. 도서관 좌석 혼잡도를 예측해 빈자리를 미리 알려 주는 서비…
1,가상대본1,2,247,"지금 도서관들은 좌석 예약 키오스크, 모바일 예약 앱, 현장 순번표 같은…"
2,가상대본1,3,278,SeatFlow의 핵심은 30분 뒤의 좌석 혼잡도를 예측하는 것입니다.\n…
3,가상대본1,4,269,"예측 모델은 제휴 도서관 3곳의 2년 치 출입 기록, 약 1,240만 건…"
4,가상대본1,5,214,이용자 화면은 세 가지로 구성했습니다.\n\n첫 화면에서는 열람실별 현재 빈…
5,가상대본1,6,215,2025년 3월부터 8주 동안 제휴 도서관 3곳에서 시범 운영을 했습니다…
6,가상대본1,7,169,"국내 공공도서관은 약 1,200곳이고, 대학 도서관은 약 430곳입니다.…"
7,가상대본1,8,160,"수익은 도서관 구독료에서 나옵니다.\n\n공공도서관은 월 49만 원, 대학 …"
8,가상대본1,9,204,앞으로의 계획입니다.\n\n1단계로 2026년 상반기까지 서울과 대전의 대학…
9,가상대본2,1,231,안녕하세요. 저희 프로젝트는 ‘남는 빵을 줄이는 동네 빵집 재고 예보’입…


## 2. Normalize Script (대본 정규화)

| 하는 것 | 하지 않는 것 |
|---|---|
| 유니코드(NFKC)·따옴표 통일 | 어미·어순·숫자 표기 바꾸기 |
| 공백·줄바꿈 정리 | 오탈자 교정 |
| **문장 분리**(Kiwi) — 문장 번호는 4장에서 LLM 에 그대로 전달되고, 이후 모든 위치(span)는 정규화 텍스트 기준 | 내용 삭제 |

발화 표현을 그대로 두는 이유: 나중에 STT 와 비교할 때 발표자가 대본 표현을 얼마나 그대로 말했는지도 보려면 원문 표현이 필요하기 때문입니다.
입력 JSON 의 `script` 는 발표자가 소리 내어 읽는 내용만 담는다고 가정합니다 (영상 재생 같은 무대 지시문 없음).

In [3]:
class Sentence(BaseModel):
    index: int
    text: str
    start: int = Field(description="정규화 텍스트 기준 시작 오프셋")
    end: int = Field(description="정규화 텍스트 기준 끝 오프셋 (exclusive)")


class NormalizedScript(BaseModel):
    text: str = Field(description="정규화된 대본. 이후 모든 span 은 이 텍스트 기준")
    sentences: list[Sentence]
    content_hash: str = Field(description="정규화 텍스트의 sha256. 대본 변경 감지용")


QUOTE_MAP = str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"', "「": '"', "」": '"'})


def normalize_script(text: str) -> NormalizedScript:
    """대본 정규화: 유니코드·따옴표 통일, 공백 정리, 문장 분리.

    발화 표현 자체(어미, 숫자 표기, 오탈자)는 바꾸지 않는다. 나중에 STT 와 대본 표현이 얼마나 같은지
    비교할 때 원문 표현이 필요하기 때문이다.
    """
    text = unicodedata.normalize("NFKC", text).translate(QUOTE_MAP)
    text = re.sub(r"\s+", " ", text).strip()

    sentences = [
        Sentence(index=i, text=s.text, start=s.start, end=s.end)
        for i, s in enumerate(kiwi.split_into_sents(text))
    ]
    return NormalizedScript(
        text=text,
        sentences=sentences,
        content_hash=hashlib.sha256(text.encode("utf-8")).hexdigest(),
    )


sample = slide_scripts["가상대본2"][0]
norm = normalize_script(sample.script)
for s in norm.sentences:
    print(f"[{s.index}] ({s.start}-{s.end}) {s.text}")

[0] (0-6) 안녕하세요.
[1] (7-43) 저희 프로젝트는 '남는 빵을 줄이는 동네 빵집 재고 예보'입니다.
[2] (44-78) 동네 빵집은 매일 아침 그날 팔릴 양을 감으로 정해 굽습니다.
[3] (79-123) 저희가 만난 빵집 사장님들은 하루 생산량의 15~20%를 폐기한다고 말했습니다.
[4] (124-161) 반대로 너무 적게 구우면 오후에 인기 빵이 품절돼 손님을 놓칩니다.
[5] (162-228) 저희는 판매 기록을 바탕으로 다음 날 품목별 판매량을 예측하고, 몇 개를 구우면 좋을지 추천하는 서비스를 만들었습니다.


## 3. Library / Rule 분기

의미 판단 없이 **객관적으로 계산 가능한 정보**를 처리합니다.

### 3-1. 한국어 수 표기 파서

`1,240만`, `1만 9천`, `2억 3천만` 같은 혼합 표기와 한글 수사(`삼십칠`, `사점육`)를 수로 바꿉니다.
대본뿐 아니라 STT 가 숫자를 한글로 받아 적는 경우에도 같은 파서를 쓸 수 있습니다.

In [4]:
HANGUL_DIGITS = {"영": 0, "공": 0, "일": 1, "이": 2, "삼": 3, "사": 4, "오": 5, "육": 6, "륙": 6, "칠": 7, "팔": 8, "구": 9}
SMALL_UNITS = {"십": 10, "백": 100, "천": 1000}
BIG_UNITS = {"만": 10**4, "억": 10**8, "조": 10**12}
NUMBER_TOKEN = re.compile(r"\d+(?:\.\d+)?|[영공일이삼사오육륙칠팔구]|[십백천]|[만억조]|점")


def parse_korean_number(s: str) -> float | None:
    """아라비아 숫자·한글 수사·혼합 표기를 수로 바꾼다.

    "1,240만" → 12400000, "1천만" → 10000000, "4.6" → 4.6,
    "천이백사십만" → 12400000, "사점육" → 4.6, "이공이육" → 2026.
    STT 는 숫자를 한글로 적는 경우가 많아서, STT 쪽 숫자를 읽을 때도 같은 함수를 쓸 수 있다.
    """
    s = s.replace(",", "").replace(" ", "")
    tokens = NUMBER_TOKEN.findall(s)
    if not tokens or "".join(tokens) != s:
        return None

    total, section, current = 0.0, 0.0, None
    fraction, fraction_scale = None, 1.0  # "점" 뒤 한글 소수부
    for tok in tokens:
        if fraction is not None:
            if tok not in HANGUL_DIGITS:
                return None
            fraction_scale /= 10
            fraction += HANGUL_DIGITS[tok] * fraction_scale
        elif tok[0].isdigit():
            current = float(tok)
        elif tok in HANGUL_DIGITS:
            # "이공이육" 처럼 단위 없이 이어지는 한글 숫자는 자릿수 나열로 본다
            current = HANGUL_DIGITS[tok] if current is None else current * 10 + HANGUL_DIGITS[tok]
        elif tok in SMALL_UNITS:
            section += (1 if current is None else current) * SMALL_UNITS[tok]
            current = None
        elif tok in BIG_UNITS:
            section += current or 0
            total += (section or 1) * BIG_UNITS[tok]
            section, current = 0.0, None
        elif tok == "점":
            fraction = 0.0
    value = total + section + (current or 0) + (fraction or 0)
    return value


def format_number(x: float) -> str:
    return f"{x:,.0f}" if float(x).is_integer() else f"{x:,.10g}"


for s in ["30", "1,240만", "1천만", "1만 9천", "24억", "4.6", "천이백사십만", "삼십", "사점육", "이공이육", "구십구"]:
    print(f"{s!r:>12} → {format_number(parse_korean_number(s))}")

        '30' → 30
    '1,240만' → 12,400,000
       '1천만' → 10,000,000
     '1만 9천' → 19,000
       '24억' → 2,400,000,000
       '4.6' → 4.6
    '천이백사십만' → 12,400,000
        '삼십' → 30
       '사점육' → 4.6
      '이공이육' → 2,026
       '구십구' → 99


### 3-2. Critical Fact Parser (핵심 사실 추출)

의미가 비슷하게 전달되더라도 **빠지면 중요한 차이가 되는 정보**(수치·날짜·이름)를 따로 뽑아 둡니다.

| type | 예 (대본 표기 → 정규형) | 방법 |
|---|---|---|
| `percentage` | `약 42퍼센트` → `42%` + 한정어 `약`, `85퍼센트 이상` → `85%` + `이상`, `15~20%` → `15%`, `20%` | 정규식 + 수 파서 + Kiwi 확인 |
| `money` | `약 24억 원` → `2,400,000,000원` + `약`, `1만 9천 원` → `19,000원` | 정규식 + 수 파서 + Kiwi 확인 |
| `quantity` | `약 1,240만 건` → `12,400,000건`, `여섯 배` → `6배`, `세 가지` → `3가지` | 정규식 + 수 파서 + Kiwi 확인 |
| `duration` / `date` / `time` | `10분 이내`, `8주 동안` → `8주` / `2026년 상반기` → `2026-H1` / `오후 6시` → `18:00` | 정규식 + Kiwi 확인 |
| `ratio` / `number` | `3:1`, `일대일` → `1:1` / `4 곱하기 6` → `4×6` | 정규식 |
| `proper_noun` | `SeatFlow`, `XGBoost`, `LightGBM`, `서울`, `대전` | 영문 정규식 + Kiwi NNP |

- 정규식은 "수 + 단위" 모양의 **후보**만 찾고, 단위가 진짜 단위인지는 **Kiwi 형태소**로 확인합니다.
  `24억 원` 의 `원` 은 독립된 형태소(의존명사)라서 인정하고, `3원칙` 의 `원` 은 `원칙` 이라는 단어의 일부라서 버립니다
  (`3프로젝트`, `3시간` 의 `시`, `두 배터리` 도 같은 방식). 그래서 "이런 단어는 빼라" 는 예외 목록을 따로 두지 않습니다.
- 패턴은 **우선순위 순서**로 적용하고, 앞 패턴이 잡은 구간은 뒤 패턴이 다시 잡지 않습니다. 영문 이름을 먼저 잡아 `GPT-5`, `A100` 의 숫자가 숫자 사실로 새지 않게 합니다.
- `1단계`, `2번째` 같은 **서수는 제외**합니다. 전달 여부를 따질 정보가 아니기 때문입니다.
- `10~20%`, `5~10억 원` 처럼 범위의 앞 수는 뒤 수의 단위를 따릅니다 (`10%`, `500,000,000원`).
- **한글로 읽은 수**도 같은 패턴이 받습니다. STT 는 숫자를 들리는 대로 적기 때문입니다:
  `사십이 퍼센트` → `42%`, `만 구천 원` → `19,000원`, `십일 점 사 퍼센트` → `11.4%`, `천이백 곳` → `1,200곳`,
  `이천이십오 년 삼 월` → `2025-03`, `오후 여섯 시 반` → `18:30`, `삼 대 일` → `3:1`, `열두 곳` → `12곳`, `스물다섯 명` → `25명`.
  한글 수는 Kiwi 가 **수사(NR)** 로 분석한 경우만 인정합니다. `이 분이 오셨다` 의 `이`(관형사), `구조 개선` 의 `구` 는 수가 아닙니다.
- 같은 사실이 여러 번 나오면 하나로 합치고 등장 위치(`spans`)만 늘립니다.
- Kiwi 고유명사는 **정밀도 우선**입니다. 문맥 속 사람 이름(`한도윤`, `오예린`)은 Kiwi 가 놓치는데, 4장 LLM 분기의 `key_terms` 가 보완합니다.
  반대로 `키오스크`, `크루아상` 같은 일반 명사를 고유명사로 잘못 잡는 경우가 남아 있습니다.

In [5]:
Importance = Literal["critical", "high", "normal"]
FactType = Literal[
    "percentage", "money", "date", "time", "duration", "quantity", "ratio", "number", "proper_noun", "term",
]


class CriticalFact(BaseModel):
    id: str = ""
    type: FactType
    value: str = Field(description="대본에 적힌 표기 그대로 (예: '약 1,240만 건')")
    normalized: str = Field(description="STT 와 비교할 정규형 (예: '12,400,000건')")
    numeric_value: float | None = None
    unit: str | None = None
    qualifier: str | None = Field(default=None, description="약·이상·이내 같은 한정어")
    spans: list[tuple[int, int]] = Field(description="정규화 텍스트 기준 등장 위치 (첫 번째가 대표)")
    sentence_indices: list[int]
    source: Literal["rule", "llm"] = "rule"
    key_point_ids: list[str] = Field(default_factory=list, description="Rubric Builder 가 연결")
    importance: Importance = Field(default="normal", description="연결된 Key Point 중 가장 높은 중요도")


# ── 정규식 빌딩 블록 ───────────────────────────────────────────
NUM = r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?"
_KGROUP = rf"(?:{NUM})(?:[십백천만억조]|{NUM})*"
KNUM = rf"{_KGROUP}(?:(?<=[만억조])\s{_KGROUP}(?<=[십백천만억조]))*"  # 1,240만 / 1천만 / 2억 3천만
PRE = r"(?:(?P<pre>약|대략|최대|최소|평균|총|거의)\s?)?"
NEG = r"(?P<neg>마이너스\s?|(?<![\w)\-−])[-−])?"
POST = r"(?:\s?(?P<post>이상|이하|이내|미만|초과|가량|정도|내외|남짓))?"
NO_ALNUM_BEFORE = r"(?<![A-Za-z0-9.,])"

# 단위 목록. "원칙"의 원, "프로젝트"의 프로처럼 단위 글자로 시작하는 일반 단어는 여기서 막지 않고
# Kiwi 형태소 경계로 걸러낸다 (_unit_is_morpheme)
PERCENT_UNIT = r"퍼센트\s?포인트|%\s?포인트|%[pP]|%|퍼센트|프로"
MONEY_UNIT = r"원|달러|\$"
DURATION_UNIT = r"초|분|시간|일|주|개월|달|년"
COUNTER_UNIT = (
    r"개교|개국|개사|개소|개|명|곳|건|회|번|배|가지|차원|대|층|권|장|편|쪽"
    r"|페이지|종|위|점|세|살|인치|인|km|kg|mg|cm|mm|ml|mAh|m|g|GB|MB|TB|KB|kWh|Wh|W|V|GHz|MHz|Hz|fps|px"
)
# 한글 수사: STT 가 숫자를 한글로 받아 적는 경우와 대본의 "두 배", "세 가지"
NATIVE_ONES = {"한": 1, "두": 2, "세": 3, "석": 3, "네": 4, "넉": 4, "다섯": 5, "여섯": 6, "일곱": 7, "여덟": 8, "아홉": 9}
NATIVE_TENS = {"열": 10, "스물": 20, "스무": 20, "서른": 30, "마흔": 40, "쉰": 50, "예순": 60, "일흔": 70, "여든": 80, "아흔": 90}
_ONES = "다섯|여섯|일곱|여덟|아홉|한|두|세|석|네|넉"
NATIVE_NUMBER = rf"(?:(?:열|스물|서른|마흔|쉰|예순|일흔|여든|아흔)(?:{_ONES})?|스무|{_ONES})"  # 열두, 스물다섯, 서른, 세


def native_value(text: str) -> int:
    tens = next((t for t in NATIVE_TENS if text.startswith(t)), None)
    return NATIVE_TENS[tens] + NATIVE_ONES.get(text[len(tens):], 0) if tens else NATIVE_ONES[text]
NATIVE_UNIT = r"배|가지|명|개월|개|곳|달|시간|주|해"
DURATION_BASE = {"달": "개월", "해": "년"}
# 한글로 읽은 수 (STT): "사십이", "만 구천", "십일 점 사", "천이백사십만". Kiwi 가 수사(NR)로 분석할 때만 인정한다
HNUM = r"(?<![가-힣])[일이삼사오육칠팔구십백천만억조]+(?:\s[일이삼사오육칠팔구십백천만억조]+)*(?:\s?점\s?[영공일이삼사오육칠팔구]+)?"
NUMBER = rf"(?:{KNUM}|{HNUM})"
HANGUL_YEAR = r"(?<![가-힣])(?:이천|천구백)[일이삼사오육칠팔구십]*"
HOUR_WORDS = {"한": 1, "두": 2, "세": 3, "네": 4, "다섯": 5, "여섯": 6, "일곱": 7, "여덟": 8, "아홉": 9,
              "열": 10, "열한": 11, "열두": 12}
MONTH_WORDS = {"일": 1, "이": 2, "삼": 3, "사": 4, "오": 5, "유": 6, "육": 6, "칠": 7, "팔": 8, "구": 9,
               "시": 10, "십": 10, "십일": 11, "십이": 12}

# 영문·숫자가 섞인 이름: SeatFlow, XGBoost, GPT-5, K-팝, v1 / 5G, 3D, 4K
LATIN_NAME = re.compile(r"(?<![A-Za-z0-9%])[A-Za-z][A-Za-z0-9]*(?:[-.&][A-Za-z0-9]+)*(?:-[가-힣]+)?")
DIGIT_NAME = re.compile(r"(?<![A-Za-z0-9.,])\d+[A-Za-z][A-Za-z0-9]*(?![A-Za-z0-9])")
LATIN_STOPWORDS = {"ai"}  # 핵심 사실로 보기엔 너무 일반적인 영문 토큰
UNIT_SYMBOLS = {"km", "kg", "mg", "cm", "mm", "ml", "mL", "m", "g", "L", "GB", "MB", "TB", "KB",
                "mAh", "kWh", "Wh", "W", "V", "Hz", "GHz", "MHz", "fps", "px", "x", "X", "p", "P"}

# 순서가 중요하다: 앞 패턴이 차지한 구간은 뒤 패턴이 다시 잡지 않는다
NUMERIC_PATTERNS: list[tuple[str, re.Pattern]] = [
    # "3단계", "2번째" 같은 서수는 전달 여부를 따질 사실이 아니라서 구간만 막는다
    ("ordinal", re.compile(NO_ALNUM_BEFORE + r"\d+\s?(?P<u>단계|번째|회차|주차|차|학년|학기)")),
    ("date", re.compile(
        NO_ALNUM_BEFORE + r"(?:"
        + r"(?P<yq>(?:19|20)\d{2})\s?년\s?(?P<q>[1-4])\s?분기"
        + r"|(?P<yh>(?:19|20)\d{2})\s?년\s?(?P<h>상|하)반기"
        + r"|(?P<ys>(?:19|20)\d{2})\s?학년도"
        + r"|'?(?P<yy>\d{2})\s?년\s?(?:(?P<q2>[1-4])\s?분기|(?P<h2>상|하)반기)"
        + r"|'(?P<yy2>\d{2})\s?년"
        + r"|(?P<y>(?:19|20)\d{2})\s?년(?:\s?(?P<m>\d{1,2})\s?월)?(?:\s?(?P<d>\d{1,2})\s?일)?"
        + r"|(?P<y2>(?:19|20)\d{2})[./-]\s?(?P<m2>\d{1,2})[./-]\s?(?P<d2>\d{1,2})\.?"
        + rf"|(?P<hyq>{HANGUL_YEAR})\s?년\s?(?P<hq>[일이삼사])\s?분기"
        + rf"|(?P<hyh>{HANGUL_YEAR})\s?년\s?(?P<hhalf>상|하)반기"
        + rf"|(?P<hy>{HANGUL_YEAR})\s?년(?:\s?(?P<hm>십일|십이|십|시|유|육|[일이삼사오칠팔구])\s?월)?"
        + r"|(?P<m3>\d{1,2})\s?월(?:\s?(?P<d3>\d{1,2})\s?일)?"
        + r"|(?P<q3>[1-4])\s?분기)"
    )),
    ("time", re.compile(
        NO_ALNUM_BEFORE
        + r"(?:(?:(?P<ampm>오전|오후)\s?)?(?P<hh>\d{1,2})\s?(?P<si>시)"
        + r"(?:\s?(?P<mi>\d{1,2})\s?분|\s?(?P<half>반))?"
        + r"|(?P<ampm2>오전|오후)\s?(?P<hh2>\d{1,2}):(?P<mi2>\d{2})"
        + r"|(?:(?P<ampm3>오전|오후)\s?)?(?<![가-힣])(?P<hh3>열한|열두|한|두|세|네|다섯|여섯|일곱|여덟|아홉|열)\s?(?P<si3>시)"
        + r"(?:\s?(?P<mi3>[일이삼사오육칠팔구십]+|\d{1,2})\s?분|\s?(?P<half3>반))?)"
    )),
    ("number", re.compile(NO_ALNUM_BEFORE + r"(?P<a>\d+)\s?(?:곱하기|[xX×*])\s?(?P<b>\d+)")),
    ("ratio", re.compile(NO_ALNUM_BEFORE + r"(?P<a>\d+)\s?(?:대|:)\s?(?P<b>\d+)(?![\d:])")),
    ("ratio_hangul", re.compile(r"(?<![가-힣])(?P<a>일|이|삼|사|오|십)\s?대\s?(?P<b>일|이|삼|사|오|십)")),
    ("range", re.compile(NO_ALNUM_BEFORE + PRE + NEG + rf"(?P<n>{NUMBER})(?P<sep>\s?(?:~|∼|에서|부터)\s?)")),
    ("percentage", re.compile(NO_ALNUM_BEFORE + PRE + NEG + rf"(?P<n>{NUMBER})\s?(?P<u>{PERCENT_UNIT})" + POST)),
    ("money", re.compile(NO_ALNUM_BEFORE + PRE + NEG + rf"(?P<n>{NUMBER})\s?(?P<u>{MONEY_UNIT})" + POST)),
    ("duration", re.compile(
        NO_ALNUM_BEFORE + PRE + rf"(?P<n>{NUMBER})\s?(?P<u>{DURATION_UNIT})" + r"(?:\s?(?:동안|간|만에))?" + POST
    )),
    ("quantity", re.compile(NO_ALNUM_BEFORE + PRE + rf"(?P<n>{NUMBER})\s?(?P<u>{COUNTER_UNIT})" + POST)),
    ("native", re.compile(PRE + rf"(?<![가-힣])(?P<nat>{NATIVE_NUMBER})\s(?P<u>{NATIVE_UNIT})" + POST)),
    ("number", re.compile(NO_ALNUM_BEFORE + PRE + NEG + rf"(?P<n>{KNUM})" + POST)),
]
TYPED_PATTERNS = {kind: p for kind, p in NUMERIC_PATTERNS if kind in ("percentage", "money", "duration", "quantity")}

# ── Kiwi 형태소 확인 ─────────────────────────────────────────
UNIT_HEAD_TAGS = {"NNB", "NNG", "NR", "SW", "SL"}  # 단위의 첫 형태소: 의존명사·명사·수사·기호·영문
UNIT_TAIL_TAGS = UNIT_HEAD_TAGS | {"XSN"}          # 뒤따르는 형태소는 접미사도 허용 (번+째)


class Morphemes:
    """Kiwi 로 한 번 분석한 형태소 목록. 정규식이 찾은 구간이 온전한 형태소인지 확인한다."""

    def __init__(self, text: str):
        self.tokens = [t for t in kiwi.tokenize(text) if t.len > 0]

    def inside(self, start: int, end: int) -> list:
        return [t for t in self.tokens if start <= t.start and t.start + t.len <= end]

    def is_whole(self, start: int, end: int, head_tags: set[str], tail_tags: set[str]) -> bool:
        """[start, end) 가 형태소 경계와 정확히 맞고, 품사가 허용 목록에 있는가.

        "3원칙" 의 '원' 은 '원칙' 형태소의 앞부분이라 False, "24억 원" 의 '원' 은 True.
        """
        tokens = self.inside(start, end)
        if not tokens or tokens[0].start != start or tokens[-1].start + tokens[-1].len != end:
            return False
        if any(t.start < start < t.start + t.len or t.start < end < t.start + t.len for t in self.tokens):
            return False
        return tokens[0].tag in head_tags and all(t.tag in tail_tags for t in tokens[1:])


def _unit_is_morpheme(m: re.Match, group: str, morph: Morphemes) -> bool:
    return morph.is_whole(*m.span(group), UNIT_HEAD_TAGS, UNIT_TAIL_TAGS)


def _hangul_number_ok(m: re.Match, morph: Morphemes) -> bool:
    """수 부분이 한글로 시작하면('사십이', '만 구천') Kiwi 가 수사(NR)로 분석했는지 본다. '점'(소수점)은 의존명사다.
    "이 분이 오셨다" 의 '이' 는 관형사라서 수가 아니다."""
    if m.group("n")[0].isdigit():
        return True
    return morph.is_whole(*m.span("n"), {"NR"}, {"NR", "NNB"})


def _starts_unit_quantity(text: str, pos: int, morph: Morphemes) -> bool:
    """pos 에서 '수 + 단위' 가 시작하는가. "20대 30대" 의 30대처럼 뒤 수에 단위가 붙으면 비율이 아니다."""
    return any(
        (um := p.match(text, pos)) is not None and _unit_is_morpheme(um, "u", morph) and _hangul_number_ok(um, morph)
        for p in TYPED_PATTERNS.values()
    )


def _confirmed_by_kiwi(kind: str, m: re.Match, text: str, morph: Morphemes) -> bool:
    """정규식 후보를 Kiwi 형태소로 확인한다. 단위 글자가 더 긴 단어의 일부면 버린다."""
    if kind == "ordinal":
        return _unit_is_morpheme(m, "u", morph)
    if kind in ("percentage", "money", "duration", "quantity"):
        return _unit_is_morpheme(m, "u", morph) and _hangul_number_ok(m, morph)
    if kind == "time":  # "3시간", "세 시간", "5시리즈" 의 시는 시각이 아니다
        return all(m.group(g) is None or _unit_is_morpheme(m, g, morph) for g in ("si", "si3"))
    if kind == "ratio":
        return not _starts_unit_quantity(text, m.start("b"), morph)
    if kind == "ratio_hangul":  # "일대일"(한 단어) 이거나 양쪽이 온전한 수사 ("삼 대 일이" 의 '이' 는 조사, "삼 대 일십" 은 아님)
        return morph.is_whole(*m.span(), {"NNG"}, set()) or all(morph.is_whole(*m.span(g), {"NR"}, {"NR"}) for g in ("a", "b"))
    if kind == "native":  # 한·두·세 … 가 관형사(MM)나 수사(NR)로 쓰였는가 ("한 대학" 은 아님)
        return _unit_is_morpheme(m, "u", morph) and morph.is_whole(*m.span("nat"), {"MM", "NR"}, {"MM", "NR"})
    return True


def _fact_fields(kind: str, value: float, unit: str | None, qualifier: str | None) -> dict:
    return dict(type=kind, normalized=f"{format_number(value)}{unit or ''}", numeric_value=value, unit=unit, qualifier=qualifier)


def _unit_of(kind: str, unit: str) -> str:
    if kind == "percentage":
        return "%p" if "포인트" in unit or unit.lower() == "%p" else "%"
    if kind == "money":
        return "달러" if unit in ("달러", "$") else "원"
    return DURATION_BASE.get(unit, unit)


def _numeric_fact(kind: str, m: re.Match) -> dict | None:
    g = m.groupdict()
    qualifier = " ".join(q for q in (g.get("pre"), g.get("post")) if q) or None
    none = dict(numeric_value=None, unit=None, qualifier=None)
    if kind == "ordinal":
        return None
    if kind == "number" and g.get("a"):
        return dict(type="number", normalized=f"{g['a']}×{g['b']}", **none)
    if kind == "ratio":
        return dict(type="ratio", normalized=f"{g['a']}:{g['b']}", **none)
    if kind == "ratio_hangul":
        return dict(type="ratio", normalized=f"{parse_korean_number(g['a']):g}:{parse_korean_number(g['b']):g}", **none)
    if kind == "date":
        year = g.get("yq") or g.get("yh") or g.get("ys") or g.get("y") or g.get("y2")
        hangul_year = g.get("hyq") or g.get("hyh") or g.get("hy")
        if hangul_year:
            year = format_number(parse_korean_number(hangul_year)).replace(",", "")
        short = g.get("yy") or g.get("yy2")
        year = year or (f"20{short}" if short else None)
        quarter = g.get("q") or g.get("q2") or g.get("q3") or (str(HANGUL_DIGITS[g["hq"]]) if g.get("hq") else None)
        half = g.get("h") or g.get("h2") or g.get("hhalf")
        if quarter or half:
            period = f"Q{quarter}" if quarter else ("H1" if half == "상" else "H2")
            return dict(type="date", normalized=f"{year}-{period}" if year else period, **none)
        month = g.get("m") or g.get("m2") or g.get("m3") or (str(MONTH_WORDS[g["hm"]]) if g.get("hm") else None)
        day = g.get("d") or g.get("d2") or g.get("d3")
        parts = [year] + [f"{int(p):02d}" for p in (month, day) if p]
        return dict(type="date", normalized="-".join(p for p in parts if p), **none)
    if kind == "time":
        hour = int(g.get("hh") or g.get("hh2")) if (g.get("hh") or g.get("hh2")) else HOUR_WORDS[g["hh3"]]
        minute_text = g.get("mi") or g.get("mi2") or g.get("mi3")
        minute = int(parse_korean_number(minute_text)) if minute_text else (30 if g.get("half") or g.get("half3") else 0)
        if (g.get("ampm") or g.get("ampm2") or g.get("ampm3")) == "오후" and hour < 12:
            hour += 12
        return dict(type="time", normalized=f"{hour:02d}:{minute:02d}", **none)
    if kind == "native":
        unit = g["u"]
        fact_kind = "duration" if unit in ("개월", "달", "시간", "주", "해") else "quantity"
        return _fact_fields(fact_kind, native_value(g["nat"]), DURATION_BASE.get(unit, unit), qualifier)

    value = parse_korean_number(g["n"])
    if value is None:
        return None
    if g.get("neg"):
        value = -value
    unit = _unit_of(kind, g["u"]) if g.get("u") else None
    return _fact_fields(kind, value, unit, qualifier)


def _range_lower_fact(m: re.Match, text: str, morph: Morphemes) -> dict | None:
    """'10~20%', '5~10억 원', '3에서 5명' 의 앞 수는 뒤 수의 단위를 따른다."""
    for kind, pattern in TYPED_PATTERNS.items():
        upper = pattern.match(text, m.end())
        if upper is None or not _unit_is_morpheme(upper, "u", morph) or not _hangul_number_ok(upper, morph):
            continue
        value = parse_korean_number(m.group("n"))
        big = re.fullmatch(rf"(?:{NUM})([만억조])", upper.group("n").replace(" ", ""))
        if value is not None and big and not re.search(r"[만억조]", m.group("n")):
            value *= BIG_UNITS[big.group(1)]  # 5~10억 → 5억
        if value is None:
            return None
        if m.group("neg"):
            value = -value
        return _fact_fields(kind, value, _unit_of(kind, upper.group("u")), m.group("pre"))
    return None


def _sentence_index(norm: NormalizedScript, pos: int) -> int:
    for s in norm.sentences:
        if s.start <= pos < s.end:
            return s.index
    return norm.sentences[-1].index if norm.sentences else 0


def _add_fact(facts: dict, norm: NormalizedScript, span: tuple[int, int], **fields) -> None:
    """(type, normalized) 가 같은 사실은 하나로 합치고 등장 위치만 늘린다."""
    key = (fields["type"], fields["normalized"])
    sent = _sentence_index(norm, span[0])
    if key in facts:
        facts[key].spans.append(span)
        facts[key].sentence_indices.append(sent)
        return
    facts[key] = CriticalFact(value=norm.text[span[0]:span[1]], spans=[span], sentence_indices=[sent], **fields)


def _take(taken: list[bool], span: tuple[int, int]) -> None:
    taken[span[0]:span[1]] = [True] * (span[1] - span[0])


def extract_latin_names(norm: NormalizedScript, facts: dict, taken: list[bool]) -> None:
    """영문·숫자 이름. 한 글자(A안, x)와 숫자 뒤 단위 기호(5,000 mAh)는 이름이 아니다."""
    text = norm.text
    for m in LATIN_NAME.finditer(text):
        word = m.group(0)
        if word.lower() in LATIN_STOPWORDS or len(word) == 1:
            continue
        if word in UNIT_SYMBOLS and re.search(r"\d\s?$", text[:m.start()]):
            continue
        _take(taken, m.span())
        _add_fact(facts, norm, m.span(), type="proper_noun", normalized=word.casefold())
    for m in DIGIT_NAME.finditer(text):
        if re.sub(r"^\d+", "", m.group(0)) in UNIT_SYMBOLS or any(taken[m.start():m.end()]):
            continue
        _take(taken, m.span())
        _add_fact(facts, norm, m.span(), type="proper_noun", normalized=m.group(0).casefold())


def extract_numeric_facts(norm: NormalizedScript, facts: dict, taken: list[bool], morph: Morphemes) -> None:
    for kind, pattern in NUMERIC_PATTERNS:
        for m in pattern.finditer(norm.text):
            if kind == "range":
                span = (m.start(), m.start("sep"))
                usable = not any(taken[span[0]:span[1]]) and _hangul_number_ok(m, morph)
                fields = _range_lower_fact(m, norm.text, morph) if usable else None
                if fields:
                    _take(taken, span)
                    _add_fact(facts, norm, span, **fields)
                continue
            start, end = m.span()
            if any(taken[start:end]) or not _confirmed_by_kiwi(kind, m, norm.text, morph):
                continue  # 버린 후보는 구간을 차지하지 않으므로 뒤 패턴이 다시 볼 수 있다
            _take(taken, (start, end))
            fields = _numeric_fact(kind, m)
            if fields:
                _add_fact(facts, norm, (start, end), **fields)


PREDICATE_SUFFIX = {"XSV", "XSA"}  # 고려+하다, 보정+하다: 명사가 동사·형용사로 쓰인 경우


def _is_standalone_proper_noun(surface: str) -> bool:
    """단어만 따로 분석해도 NNP 한 덩어리로 나오는지. 문맥 때문에 NNP 로 잘못 붙은 일반 명사를 걸러낸다."""
    best = kiwi.analyze(surface, top_n=1)[0][0]
    return len(best) == 1 and best[0].tag == "NNP"


def extract_korean_proper_nouns(norm: NormalizedScript, facts: dict, taken: list[bool], morph: Morphemes) -> None:
    """Kiwi 형태소 분석의 고유명사(NNP) 태그.

    - NNP 로 시작해 띄어쓰기 없이 붙은 명사까지 한 이름으로 본다 (서울+시립+도서관)
    - 뒤에 '-하다'가 붙으면 고유명사가 아니다 (보정하는)
    - NNP 한 토큰뿐이면 단독 분석으로 한 번 더 확인한다

    정밀도 우선이다. 사람 이름처럼 Kiwi 가 놓치는 고유명사는 LLM 분기의 key_terms 가 보완한다.
    """
    tokens = morph.tokens
    i = 0
    while i < len(tokens):
        if tokens[i].tag != "NNP":
            i += 1
            continue
        j = i + 1
        while j < len(tokens) and tokens[j].tag in ("NNP", "NNG", "SL") and tokens[j].start == tokens[j - 1].end:
            j += 1
        run, nxt = tokens[i:j], tokens[j] if j < len(tokens) else None
        i = j
        span = (run[0].start, run[-1].end)
        surface = norm.text[span[0]:span[1]]
        if len(surface) < 2 or any(taken[span[0]:span[1]]):
            continue
        if nxt is not None and nxt.tag in PREDICATE_SUFFIX and nxt.start == span[1]:
            continue
        if len(run) == 1 and not _is_standalone_proper_noun(surface):
            continue
        _add_fact(facts, norm, span, type="proper_noun", normalized=surface)


def extract_critical_facts(norm: NormalizedScript) -> list[CriticalFact]:
    """Critical Fact Parser: 숫자·비율·날짜·시각·기간·금액·고유명사를 규칙으로 추출한다."""
    facts: dict = {}
    taken = [False] * len(norm.text)
    morph = Morphemes(norm.text)  # 한 번 분석해서 숫자 단위 확인과 고유명사 추출에 같이 쓴다
    extract_latin_names(norm, facts, taken)   # GPT-5, A100 안의 숫자를 숫자 사실로 잡지 않도록 먼저
    extract_numeric_facts(norm, facts, taken, morph)
    extract_korean_proper_nouns(norm, facts, taken, morph)

    ordered = sorted(facts.values(), key=lambda f: f.spans[0][0])
    for i, fact in enumerate(ordered, 1):
        fact.id = f"CF{i}"
    return ordered


def facts_table(facts: list[CriticalFact]) -> pd.DataFrame:
    return pd.DataFrame([
        {"id": f.id, "type": f.type, "value": f.value, "normalized": f.normalized,
         "qualifier": f.qualifier, "count": len(f.spans), "sentences": f.sentence_indices}
        for f in facts
    ])


# 예시 문장
demo = normalize_script("2026년까지 서울시립도서관에서 GPT-5를 활용해 3개월 동안 실험했고, 사용자는 전년 대비 37% 증가했습니다.")
display(facts_table(extract_critical_facts(demo)))

# 가상 데이터: 가상대본1 슬라이드 6
facts_table(extract_critical_facts(normalize_script(slide_scripts["가상대본1"][5].script)))

,id,type,value,normalized,qualifier,count,sentences
0,CF1,date,2026년,2026,None,1,[0]
1,CF2,proper_noun,서울시립도서관,서울시립도서관,None,1,[0]
2,CF3,proper_noun,GPT-5,gpt-5,None,1,[0]
3,CF4,duration,3개월 동안,3개월,None,1,[0]
4,CF5,percentage,37%,37%,None,1,[0]


,id,type,value,normalized,qualifier,count,sentences
0,CF1,date,2025년 3월,2025-03,NaN,1,[0]
1,CF2,duration,8주 동안,8주,NaN,1,[0]
2,CF3,quantity,3곳,3곳,NaN,1,[0]
3,CF4,quantity,"1,860명","1,860명",NaN,1,[1]
4,CF5,duration,평균 23분,23분,평균,1,[1]
5,CF6,duration,13분,13분,NaN,1,[1]
6,CF7,percentage,약 42퍼센트,42%,약,1,[1]
7,CF8,duration,10분,10분,NaN,1,[2]
8,CF9,percentage,76%,76%,NaN,1,[2]
9,CF10,percentage,18%,18%,NaN,1,[3]


### 3-3. TF-IDF / Keyword (키워드 중요도)

- **문서 집합 = 한 발표의 슬라이드들.** 여러 슬라이드에 두루 나오는 말(`예약`, `빵집`)은 IDF 가 낮아지고, 이 슬라이드에서만 두드러지는 말이 올라옵니다.
  다만 표지처럼 짧은 슬라이드에서는 두루 나오는 말도 상위에 남을 수 있습니다.
- 토큰은 Kiwi 명사 + 붙어 있는 명사 2-gram(`시범 운영`, `추천 생산량`). `고려해`, `활용하여`처럼 동사로 쓰인 명사는 버립니다.
- 발표 전체가 필요하므로 파이프라인은 슬라이드를 한 장씩이 아니라 **JSON 파일 단위로** 받습니다 (8장).

In [6]:
class Keyword(BaseModel):
    term: str
    score: float = Field(description="TF-IDF 점수 (슬라이드 안에서 L2 정규화)")
    tf: int = Field(description="슬라이드 안 등장 횟수")


NOUN_TAGS = {"NNG", "NNP", "SL", "SH"}
KEYWORD_STOPWORDS = {
    "저희", "여기", "이번", "이후", "다음", "먼저", "경우", "부분", "정도", "마지막",
    "이상", "감사", "안녕", "발표", "말씀", "자체", "하나", "우리", "자신",
}


def tokenize_nouns(text: str) -> list[str]:
    """TF-IDF 용 토큰: 명사 단위 + 붙어 있는 명사 2-gram ("좌석 예약", "예측 오차").

    - 명사+접미사(교수+자, 사용+자)는 한 단어로 합친다
    - 이어 붙은 영문·숫자(A+100, 5+G)는 한 단어로 합친다
    - 뒤에 '-하다'가 붙어 동사로 쓰인 명사(고려해, 활용하여)는 버린다
    - 한 글자 한글 명사(방, 곳, 양)는 뜻이 약해서 버린다
    """
    words: list[tuple[str, int, int]] = []  # (form, start, end)
    for tok in kiwi.tokenize(text):
        prev = words[-1] if words else None
        joinable = prev is not None and prev[2] == tok.start
        if tok.tag in PREDICATE_SUFFIX and joinable:
            words[-1] = ("", prev[1], tok.start + tok.len)
        elif tok.tag == "XSN" and joinable and prev[0]:
            words[-1] = (prev[0] + tok.form, prev[1], tok.start + tok.len)
        elif tok.tag in ("SL", "SN") and joinable and re.fullmatch(r"[A-Za-z0-9]+", prev[0]):
            words[-1] = (prev[0] + tok.form, prev[1], tok.start + tok.len)
        elif tok.tag in NOUN_TAGS or tok.tag == "SN":  # 숫자는 5G, 3D 처럼 뒤 영문과 합치려고 둔다
            words.append((tok.form, tok.start, tok.start + tok.len))
        else:
            words.append(("", tok.start, tok.start + tok.len))  # 명사 연쇄를 끊는 표지

    def keep(w: str) -> bool:
        return len(w) >= 2 and w not in KEYWORD_STOPWORDS and not re.fullmatch(r"[\d.,]+", w)

    unigrams = [w for w, _, _ in words if keep(w)]
    bigrams = [
        f"{a[0]} {b[0]}" for a, b in zip(words, words[1:])
        if keep(a[0]) and keep(b[0]) and b[1] - a[2] <= 1  # 사이에 조사 없이 붙거나 한 칸 띄운 명사
    ]
    return unigrams + bigrams


def extract_keywords_tfidf(norms: list[NormalizedScript], top_k: int = 8) -> list[list[Keyword]]:
    """TF-IDF / Keyword: 한 발표의 슬라이드들을 문서 집합으로 보고 슬라이드별 상위 키워드를 뽑는다.

    IDF 가 발표 안에서 계산되므로 "여러 슬라이드에 두루 나오는 말"은 낮고
    "이 슬라이드에서만 두드러지는 말"은 높게 나온다.
    """
    docs = [n.text for n in norms]
    vectorizer = TfidfVectorizer(analyzer=tokenize_nouns, sublinear_tf=True, norm="l2")
    try:
        matrix = vectorizer.fit_transform(docs)
    except ValueError:  # 발표 전체에 명사가 하나도 없음
        return [[] for _ in docs]
    vocab = vectorizer.get_feature_names_out()

    results = []
    for i, doc in enumerate(docs):
        counts = Counter(tokenize_nouns(doc))
        row = matrix.getrow(i).toarray().ravel()
        top = row.argsort()[::-1][:top_k]
        results.append([
            Keyword(term=vocab[j], score=round(float(row[j]), 4), tf=counts[vocab[j]])
            for j in top if row[j] > 0
        ])
    return results


deck_norms = [normalize_script(s.script) for s in slide_scripts["가상대본2"]]
for slide, kws in zip(slide_scripts["가상대본2"], extract_keywords_tfidf(deck_norms)):
    print(slide.slide_number, [f"{k.term}({k.score:.2f})" for k in kws])

1 ['동네 빵집(0.33)', '동네(0.33)', '빵집(0.27)', '하루 생산량(0.19)', '아침 그날(0.19)', '아침(0.19)', '사장님들(0.19)', '빵집 사장님들(0.19)']
2 ['팀원(0.38)', '화면 개발(0.18)', '한도윤(0.18)', '한도윤 팀원(0.18)', '정리(0.18)', '팀원별(0.18)', '팀원별 역할(0.18)', '서민재 팀원(0.18)']
3 ['형식(0.28)', '행사(0.24)', '데이터(0.21)', '판매(0.19)', '빵집(0.19)', '기록(0.17)', '판매 기록(0.17)', '포스기(0.16)']
4 ['기간(0.32)', '판매량 규모(0.19)', 'Prophet(0.19)', '평가(0.19)', '시계열(0.19)', '교차(0.19)', '기본(0.19)', 'LightGBM(0.19)']
5 ['기존(0.36)', '품목(0.31)', '오차(0.27)', '폐기율(0.21)', '기존 품목(0.21)', '방식(0.21)', 'MAPE 기준(0.21)', '기존 방식(0.21)']
6 ['손해(0.41)', '원가(0.41)', '생산량(0.28)', '주문(0.24)', '이득(0.24)', '단체(0.24)', '단체 주문(0.24)', '보수적(0.24)']
7 ['추천(0.34)', '추천 생산량(0.34)', '생산량(0.26)', '에이전트(0.26)', '날씨(0.26)', '이유(0.23)', '날씨 정보(0.23)', '날씨 예보(0.23)']
8 ['할인(0.34)', '리포트(0.27)', '에이전트(0.21)', '할인 정보(0.19)', '조언(0.19)', '지난주(0.19)', '지난주 판매량(0.19)', '생산(0.19)']
9 ['폐기(0.48)', '요금(0.39)', '비용(0.39)', '효과(0.23)', '제휴 매장(0.23)', '유료(0.23)', '매장(0.23)', '목표(0.23)']
10 ['생산량(

## 4. LLM API 1차 분석 (의미 분석)

의미 이해가 필요한 **Key Point Extraction · Importance · Core Claim** 을 structured output 한 번의 호출로 받습니다.
입력은 2장에서 나눈 문장에 번호(`[S0]`, `[S1]` …)를 붙인 대본이고, LLM 은 **중요한 문장을 먼저 검사한 뒤** 결과를 만듭니다.
structured output 은 스키마 필드 순서대로 생성되므로, 필드 순서가 곧 작업 순서입니다.

| 순서 | 필드 | 뜻 |
|---|---|---|
| 1 | `sentence_roles` | **모든 문장의 역할.** `claim`(핵심 주장, 슬라이드당 정확히 1문장 · 팀 소개처럼 나열만 하는 슬라이드는 0) / `evidence`(빠지면 핵심 주장이 설득력을 잃는 근거·수치) / `detail`(예시·배경) / `skip`(인사·전환) |
| 2 | `core_claim` | claim 문장을 바탕으로 정리한 핵심 주장 한 문장 |
| 3 | `key_points[].content` | 발표자가 전달해야 하는 핵심 내용 한 문장. 하나에 주장·사실 하나만 |
| 3 | `key_points[].sentence_ids` | 근거 문장 **번호**. skip 이 아닌 모든 문장이 어느 Key Point 에든 들어가야 합니다 |
| 3 | `key_points[].key_terms` | 전달했다고 보려면 꼭 말해야 하는 이름·전문 용어. 규칙이 놓친 고유명사(사람 이름 등)를 보완합니다 |

이렇게 한 이유:
- **Key Point 의 경계를 문장 단위로 고정합니다.** LLM 이 대본을 자유롭게 나누게 하면 같은 입력이라도 호출마다 나누는 방식이 달라져 채점 기준이 흔들립니다.
- **중요도는 LLM 이 직접 고르지 않습니다.** 문장 역할만 표시하게 하고, Key Point 의 중요도는 5장에서 코드가 근거 문장의 역할로 정합니다.
- **근거 문장을 복사하지 않고 번호로 가리킵니다.** 복사하다 한두 글자가 틀려 근거 위치를 잃는 일이 없습니다.
- 대신 Key Point 는 문장보다 작아질 수 없습니다. 한 문장에 내용이 둘이면 한 Key Point 로 묶입니다.

LLM 에게 Critical Fact 를 직접 고르게 하지 않는 이유: 두 분기가 병렬이라 LLM 은 규칙 분기 결과를 모르고, 없는 사실을 지어낼 수도 있기 때문입니다.
사실과 Key Point 의 연결은 5장에서 코드가 합니다.

In [7]:
SentenceRole = Literal["claim", "evidence", "detail", "skip"]


class SentenceLabel(BaseModel):
    id: int = Field(description="문장 번호. 입력의 [S번호]")
    role: SentenceRole = Field(description="claim | evidence | detail | skip")


class KeyPointDraft(BaseModel):
    content: str = Field(description="핵심 내용 한 문장. '~이다/~한다'로 끝나는 평서문. 대본의 수치·고유명사는 그대로 유지")
    sentence_ids: list[int] = Field(description="근거 문장 번호. 이어지는 문장 1개 이상")
    key_terms: list[str] = Field(description="이 핵심 내용을 전달하려면 반드시 말해야 하는 이름·용어. 대본 표기 그대로. 0~3개")


class SlideSemanticAnalysis(BaseModel):
    # 필드 순서가 생성 순서다: 문장 역할을 먼저 모두 정한 뒤 그것을 바탕으로 Key Point 를 만든다
    sentence_roles: list[SentenceLabel] = Field(description="모든 문장의 역할. 입력 문장마다 하나씩")
    core_claim: str = Field(description="이 슬라이드가 청중에게 남기려는 핵심 주장 한 문장")
    key_points: list[KeyPointDraft] = Field(description="발표자가 반드시 전달해야 하는 핵심 내용 목록")


SEMANTIC_SYSTEM_PROMPT = """
너는 발표 코칭 서비스의 대본 분석가야.
입력은 발표 대본 중 슬라이드 한 장이고, 문장마다 [S번호] 가 붙어 있어. 네 결과는 나중에 발표자의 실제 발화(STT)와 비교해서
핵심 내용이 전달됐는지 채점하는 기준(rubric)으로 쓰여. 아래 순서대로 작업해.

# 1단계: sentence_roles — 모든 문장의 역할을 먼저 표시해
- 입력의 모든 문장에 빠짐없이 하나씩 표시해.
- claim: 이 슬라이드가 청중에게 남기려는 핵심 주장을 가장 직접적으로 말하는 문장. 슬라이드당 정확히 1문장.
  단, 팀 소개·목차처럼 대등한 항목을 나열하기만 하는 슬라이드에는 claim 을 두지 마.
- evidence: claim 을 뒷받침하는 근거·기능·수치. 빠지면 claim 이 설득력을 잃는 문장.
- detail: 예시·배경·부연 설명. 빠져도 claim 은 그대로 전달되는 문장.
- skip: 인사, 전환 멘트("다음으로 ~입니다"), 마무리 인사("감사합니다")처럼 전달 여부를 따질 필요가 없는 문장.

# 2단계: core_claim
- claim 문장을 바탕으로 이 슬라이드의 핵심 주장을 한 문장으로 정리해.

# 3단계: key_points
- skip 이 아닌 문장으로 key point 를 만들어. skip 이 아닌 모든 문장이 어느 key point 에든 들어가야 해.
- key point 하나에는 주장·사실 하나만 담고, 서로 독립적으로 전달 여부를 판단할 수 있어야 해.
- sentence_ids 에는 근거 문장 번호를 적어. 이어지는 문장만 묶고, 같은 내용을 다르게 말한 이어지는 문장은 하나로 묶어.
- 대본 길이에 맞춰 2~6개를 만들어.
- content 는 '~이다', '~한다'로 끝나는 평서문 한 문장으로 써. 대본의 수치, 고유명사, 전문 용어는 바꾸지 말고 그대로 쓰고, 대본에 없는 내용은 추가하지 마.

# key_terms 규칙
- 이 key point를 전달했다고 인정하려면 발표자가 반드시 말해야 하는 이름·용어만 0~3개 골라.
- 넣는 것: 서비스·제품·팀·사람·기관·모델 이름, 이 분야에서만 쓰는 전문 용어 (예: 서비스 이름, 발표자 이름, BERT, 강화학습)
- 넣지 않는 것: 다른 말로 바꿔 말해도 뜻이 통하는 일반 명사 (예: 사용자, 서비스, 데이터, 기능, 화면, 결과)
- 대본에 적힌 표기 그대로 써. 숫자·금액·비율은 규칙 기반으로 따로 추출하니까 넣지 마.
"""

USER_MESSAGE_TEMPLATE = "[슬라이드 {slide_number}]\n{script}"
SENTENCE_LINE = "[S{index}] {text}"


def llm_config_hash() -> str:
    """프롬프트·메시지 형식·스키마·모델이 바뀌면 달라지는 해시. 같은 대본이라도 이 값이 다르면 다시 분석한다."""
    payload = json.dumps({
        "model": MODEL,
        "prompt": SEMANTIC_SYSTEM_PROMPT,
        "user_message": USER_MESSAGE_TEMPLATE,
        "sentence_line": SENTENCE_LINE,
        "schema": SlideSemanticAnalysis.model_json_schema(),
    }, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


def build_semantic_llm():
    """SlideSemanticAnalysis 모양으로만 답하도록 고정한 LLM."""
    model = ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, model=MODEL, max_retries=2)
    return model.with_structured_output(SlideSemanticAnalysis)


def semantic_user_message(slide: SlideScript, norm: NormalizedScript) -> str:
    """정규화 단계에서 나눈 문장에 번호를 붙여 보낸다. LLM 은 문장을 복사하지 않고 번호로 가리킨다."""
    numbered = "\n".join(SENTENCE_LINE.format(index=s.index, text=s.text) for s in norm.sentences)
    return USER_MESSAGE_TEMPLATE.format(slide_number=slide.slide_number, script=numbered)


def analyze_semantics(slide: SlideScript, norm: NormalizedScript, llm) -> SlideSemanticAnalysis:
    """LLM API (의미 분석): 문장 역할 → Core Claim → Key Point 를 한 번의 호출로 받는다."""
    return llm.invoke([
        ("system", SEMANTIC_SYSTEM_PROMPT),
        ("user", semantic_user_message(slide, norm)),
    ])


print("llm_config_hash:", llm_config_hash())
print(semantic_user_message(slide_scripts["가상대본1"][0], normalize_script(slide_scripts["가상대본1"][0].script)))

llm_config_hash: f7b2ef94d4e7
[슬라이드 1]
[S0] 안녕하십니까.
[S1] 도서관 좌석 혼잡도를 예측해 빈자리를 미리 알려 주는 서비스 SeatFlow를 만든 빈자리연구소입니다.
[S2] 시험 기간이 되면 열람실은 늘 붐빕니다.
[S3] 저희가 대학생 312명에게 물어보니, 빈자리를 찾느라 하루 평균 23분을 쓴다고 답했습니다.
[S4] 자리가 없는 것만이 문제가 아닙니다.
[S5] 가방만 두고 자리를 비우는 경우가 많아서, 실제로는 비어 있는 좌석도 찾기 어렵습니다.
[S6] 그래서 저희는 좌석이 언제 비는지 미리 알려 주면 이 시간을 줄일 수 있다고 생각했습니다.


## 5. 1차 결과 정리·검증 (코드)

규칙 기반 분석과 LLM 1차 분석을 합치고 문제를 찾는 단계이고, **LLM 을 쓰지 않습니다.** 여기서 만든 초안과 경고가 6장 최종 결론의 입력이 됩니다.

1. `resolve_roles` · `make_key_points` — `sentence_ids` 로 근거 원문·위치를 정하고, 근거 문장 역할 중 가장 높은 것으로 중요도를 정합니다
   (`claim` → critical, `evidence` → high, `detail` → normal). 범위를 벗어난 번호는 버리고 경고합니다
2. `merge_key_terms` — LLM `key_terms` 중 규칙이 못 잡은 것을 `source="llm"` 인 `term` 사실로 추가. 대본에 없는 용어는 버리고 경고.
   이름·용어가 더 긴 이름·용어 안에 들어 있으면 지워서 **한 발화가 한 번만** 채점되게 합니다 (`서울·대전 권역` 안의 `서울`).
   바꿔 말해도 되는 용어는 초안에서 사실로 올리지 않고 경고합니다 (`key_term_rejection`: 4어절 이상의 구절, 같은 발표의 2개 이상 슬라이드에 나오는 일반 명사).
   이 판정은 6장 최종 결론 LLM 에 참고 정보로도 넘어갑니다
3. `link_facts_to_key_points` — Key Point 가 **그 사실을 직접 말할 때** 연결합니다: key_terms 로 지목했거나, 근거 구간 안에 나오는 사실을 content 가 같은 값으로 말한 경우.
   근거 위치만으로 연결하면 한 문장을 나눠 가진 Key Point 들이 서로의 숫자를 떠안기 때문입니다. 아무도 말하지 않는 사실은 근거 구간이 그것을 품은 Key Point 가 하나뿐일 때만 연결합니다.
   사실의 `importance` 는 연결된 Key Point 중 가장 높은 것을 물려받습니다 (사실별로 채점 가중치를 다르게 줄 수 있게). content 에 **대본에 없는 수치**가 있으면(LLM 환각) 경고합니다
4. `validate_rubric` — claim 문장 수, Key Point 에 안 들어간 문장, critical 개수, 어느 Key Point 에도 안 걸린 사실을 `warnings` 로 남깁니다

In [8]:
class KeyPoint(BaseModel):
    id: str
    content: str
    importance: Importance = Field(description="근거 문장의 역할로 코드가 정한다 (ROLE_IMPORTANCE)")
    sentence_indices: list[int] = Field(description="근거 문장 번호 (LLM 이 번호로 지정)")
    source_quote: str = Field(description="근거 문장들의 원문")
    source_span: tuple[int, int] | None = Field(description="근거 문장들의 위치. 유효한 번호가 없으면 None")
    key_terms: list[str]
    fact_ids: list[str] = Field(description="이 Key Point 가 직접 말하는 Critical Fact")


class RubricMeta(BaseModel):
    rubric_id: str = Field(description="대본·LLM 설정·스키마가 같으면 같은 값. 평가 결과가 어떤 기준으로 채점됐는지 가리킨다")
    script_name: str = Field(description="입력 JSON 파일 이름 (확장자 제외)")
    slide_number: int
    content_hash: str
    rubric_schema_version: str
    llm_model: str
    llm_config_hash: str = Field(description="1차 분석 LLM 설정 해시")
    final_config_hash: str = Field(default="", description="최종 결론 LLM 설정 해시. 1차 결과(초안)면 빈 값")
    created_at: str


class NameDecision(BaseModel):
    value: str
    source: Literal["rule", "llm"]
    keep: bool = Field(description="발표자가 이 말을 그대로 해야 하는가 (True 면 Critical Fact)")
    reason: str


class EvaluationRubric(BaseModel):
    meta: RubricMeta
    normalized_script: str
    sentences: list[Sentence]
    sentence_roles: list[str] = Field(description="문장별 역할 (claim / evidence / detail / skip). sentences 와 순서가 같다")
    core_claim: str
    key_points: list[KeyPoint]
    critical_facts: list[CriticalFact]
    keywords: list[Keyword]
    warnings: list[str]
    draft_warnings: list[str] = Field(default_factory=list, description="1차 결과에 대한 코드 검증 경고 (최종 결론의 입력)")
    review_changes: list[str] = Field(default_factory=list, description="최종 결론이 1차 결과에서 바꾼 점")
    name_decisions: list[NameDecision] = Field(default_factory=list, description="이름·용어 후보별 최종 결정")


IMPORTANCE_RANK = {"normal": 0, "high": 1, "critical": 2}
# Key Point 의 중요도 = 근거 문장 역할 중 가장 높은 것
ROLE_IMPORTANCE = {"claim": "critical", "evidence": "high", "detail": "normal", "skip": "normal"}
# 채점할 때 Key Point·사실의 가중치 (STT 평가 노트북의 점수 계산과 12장 일관성 측정에서 쓴다)
SCORE_WEIGHT = {"critical": 3, "high": 2, "normal": 1}


def importance_from_roles(roles: list[str]) -> str:
    """근거 문장 역할로 중요도를 정한다. LLM 이 중요도를 직접 고르지 않게 해서 호출마다 흔들리는 폭을 줄인다."""
    return max((ROLE_IMPORTANCE[r] for r in roles), key=IMPORTANCE_RANK.get, default="normal")


def _compact(text: str) -> tuple[str, list[int]]:
    """공백·문장부호를 뺀 문자열과, 각 글자의 원래 위치."""
    chars, index = [], []
    for i, ch in enumerate(text):
        if ch.isalnum():
            chars.append(ch.lower())
            index.append(i)
    return "".join(chars), index


def _overlaps(a: tuple[int, int], b: tuple[int, int]) -> bool:
    return a[0] < b[1] and b[0] < a[1]


def _covers(outer: tuple[int, int], inner: tuple[int, int]) -> bool:
    return outer[0] <= inner[0] and inner[1] <= outer[1]


def _term_spans(term: str, text: str) -> list[tuple[int, int]]:
    """용어 등장 위치. 영문 용어는 더 긴 영문 단어 안에서 잡지 않는다 (GPT ⊄ GPTs)."""
    pattern = re.escape(term)
    if re.match(r"[A-Za-z0-9]", term):
        pattern = r"(?<![A-Za-z0-9])" + pattern
    if re.search(r"[A-Za-z0-9]$", term):
        pattern += r"(?![A-Za-z0-9])"
    return [m.span() for m in re.finditer(pattern, text, flags=re.IGNORECASE)]


NAME_TYPES = ("proper_noun", "term")


def _drop_nested(facts: list[CriticalFact]) -> list[CriticalFact]:
    """이름·용어 사실이 더 긴 이름·용어 안에 들어 있으면 그 위치를 지운다. 한 발화가 여러 번 채점되지 않게 가장 긴 것만 남긴다.
    (서울·대전 권역 안의 서울·대전 / 시계열 교차검증 안의 교차검증)
    수치 사실은 값으로 따로 검증하므로 용어 안에 있어도 지우지 않는다 ('2년 치 기록' 안의 2년)."""
    names = [f for f in facts if f.type in NAME_TYPES]
    kept = []
    for fact in facts:
        if fact.type not in NAME_TYPES:
            kept.append(fact)
            continue
        keep = [
            i for i, s in enumerate(fact.spans)
            if not any(o != s and _covers(o, s) for other in names if other is not fact for o in other.spans)
        ]
        if keep:
            fact.spans = [fact.spans[i] for i in keep]
            fact.sentence_indices = [fact.sentence_indices[i] for i in keep]
            kept.append(fact)
    return kept


def key_term_rejection(term: str, deck_texts: list[str]) -> str | None:
    """LLM key_term 을 Critical Fact 로 올리지 않을 이유. 올려도 되면 None.

    Critical Fact 는 STT 에 그 말이 그대로 나왔는지로 채점하므로, 다른 말로 바꿔 말해도 되는 말은 올리지 않는다.
    프롬프트로도 막지만 LLM 이 지키지 않을 때가 있어서 코드로 한 번 더 거른다.
    - 영문·숫자가 들어간 이름(SeatFlow, XGBoost)이나 Kiwi 가 고유명사(NNP)로 보는 말은 통과
    - 4어절 이상이면 용어가 아니라 구절(프로젝트 제목 등)이다: 글자 그대로 말하기를 기대하기 어렵다
    - 같은 발표의 2개 이상 슬라이드에 나오는 일반 명사는 이 발표의 공통 어휘다 (에이전트, 주간 리포트)
    """
    if len(term.split()) >= 4:
        return "phrase"
    if re.search(r"[A-Za-z0-9]", term) or any(t.tag == "NNP" for t in kiwi.tokenize(term)):
        return None
    if sum(bool(_term_spans(term, text)) for text in deck_texts) >= 2:
        return "common_in_deck"
    return None


def merge_key_terms(
    key_points: list[KeyPoint], facts: list[CriticalFact], norm: NormalizedScript,
    deck_texts: list[str], warnings: list[str], filter_terms: bool = True,
) -> list[CriticalFact]:
    """LLM 의 key_terms 를 Critical Fact 로 합친다.

    이미 있는 사실이 덮지 않는 등장 위치가 남으면 source='llm' 인 term 으로 추가하고 (좌석 예약 시스템 ⊋ 좌석 예약),
    모든 위치가 덮여 있으면 그 사실을 그대로 쓴다 (SeatFlow = SeatFlow).
    대본에 없는 용어는 LLM 이 만들어 낸 것이므로 버리고 경고를 남긴다.
    바꿔 말해도 되는 용어(key_term_rejection)는 Key Point 의 key_terms 에는 남기되 사실로는 올리지 않는다.
    filter_terms=False 면 거르지 않는다 (최종 결론에 넘길 이름·용어 후보를 모을 때).
    """
    facts = list(facts)
    for kp in key_points:
        for term in kp.key_terms:
            spans = _term_spans(term, norm.text)
            if not spans:
                warnings.append(f"key_term_not_found:{kp.id}:{term}")
                continue
            reason = key_term_rejection(term, deck_texts) if filter_terms else None
            if reason:
                warnings.append(f"key_term_not_fact:{kp.id}:{term}({reason})")
                continue
            fresh = [s for s in spans if not any(_covers(fs, s) for f in facts for fs in f.spans)]
            if fresh:
                facts.append(CriticalFact(
                    type="term", value=term, normalized=term.casefold(), spans=fresh,
                    sentence_indices=[_sentence_index(norm, s[0]) for s in fresh], source="llm",
                ))
    facts = sorted(_drop_nested(facts), key=lambda f: f.spans[0][0])
    for i, fact in enumerate(facts, 1):
        fact.id = f"CF{i}"
    return facts


def _numeric_mentions(text: str) -> set[tuple[str, str]]:
    """문장에 나온 수치 사실의 (type, 정규형). Key Point content 가 어떤 수치를 말하는지 볼 때 쓴다."""
    return {
        (f.type, f.normalized) for f in extract_critical_facts(normalize_script(text))
        if f.type not in ("proper_noun", "term")
    }


def _says(kp: KeyPoint, fact: CriticalFact, kp_numbers: set[tuple[str, str]]) -> bool:
    """Key Point 가 이 사실을 직접 말하는가.

    - key_terms 로 직접 지목했으면 말한 것이다
    - content 에 같은 수치·이름이 있고, 그 사실이 Key Point 근거 구간 안에 나오면 말한 것이다
      (근거 구간 조건이 없으면 '서울시립도서관' 을 말하는 KP 에 '서울' 이 붙는다)
    """
    value = _compact(fact.value)[0]
    if any(_compact(t)[0] == value for t in kp.key_terms):
        return True
    if kp.source_span is not None and not any(_overlaps(s, kp.source_span) for s in fact.spans):
        return False
    if fact.type not in ("proper_noun", "term"):
        return (fact.type, fact.normalized) in kp_numbers
    return value in _compact(kp.content)[0] or any(value in _compact(t)[0] for t in kp.key_terms)


def link_facts_to_key_points(key_points: list[KeyPoint], facts: list[CriticalFact], warnings: list[str]) -> None:
    """Critical Fact ↔ Key Point 연결 (결정적).

    1) Key Point 가 그 사실을 직접 말하면 연결한다 (`_says`).
       한 문장을 두 KP 가 나눠 가져도, '약 1,240만 건으로 학습' 을 말한 KP 에만 숫자가 붙고 다른 KP 에는 안 붙는다.
    2) 어느 Key Point 도 직접 말하지 않는 사실은, 근거 구간이 그 사실을 품은 Key Point 가 하나뿐일 때만 연결한다.
    사실의 중요도는 연결된 Key Point 중 가장 높은 것을 물려받는다.
    content 에 대본에 없는 수치가 있으면 LLM 이 만들어 낸 것이므로 경고를 남긴다.
    """
    kp_numbers = {kp.id: _numeric_mentions(kp.content) for kp in key_points}
    script_numbers = {(f.type, f.normalized) for f in facts}
    for kp in key_points:
        kp.fact_ids = []
        for _, value in sorted(kp_numbers[kp.id] - script_numbers):
            warnings.append(f"unsupported_number_in_key_point:{kp.id}:{value}")

    for fact in facts:
        fact.key_point_ids, fact.importance = [], "normal"
        linked = [kp for kp in key_points if _says(kp, fact, kp_numbers[kp.id])]
        if not linked:
            holders = [kp for kp in key_points if kp.source_span and any(_overlaps(s, kp.source_span) for s in fact.spans)]
            linked = holders if len(holders) == 1 else []
        for kp in linked:
            kp.fact_ids.append(fact.id)
            fact.key_point_ids.append(kp.id)
            if IMPORTANCE_RANK[kp.importance] > IMPORTANCE_RANK[fact.importance]:
                fact.importance = kp.importance


def validate_rubric(key_points: list[KeyPoint], facts: list[CriticalFact], roles: list[str], warnings: list[str]) -> None:
    """LLM 출력이 채점 기준으로 쓸 만한지 규칙으로 점검한다. 문제는 고치지 않고 경고로 남긴다."""
    n_claims = roles.count("claim")
    if n_claims > 1:
        warnings.append(f"too_many_claims:{n_claims}")
    covered = {i for kp in key_points for i in kp.sentence_indices}
    uncovered = [i for i, role in enumerate(roles) if role != "skip" and i not in covered]
    if uncovered:
        warnings.append(f"uncovered_sentences:{','.join(map(str, uncovered))}")
    n_critical = sum(kp.importance == "critical" for kp in key_points)
    if not key_points:
        warnings.append("no_key_points")
    elif n_critical == 0:
        warnings.append("no_critical_key_point")  # 팀 소개처럼 나열만 하는 슬라이드라면 정상
    elif n_critical > 1:
        warnings.append(f"too_many_critical:{n_critical}")
    if len(key_points) > 6:
        warnings.append(f"too_many_key_points:{len(key_points)}")
    for kp in key_points:
        if kp.source_span is None:
            warnings.append(f"key_point_without_sentence:{kp.id}")
    unlinked = [f.id for f in facts if not f.key_point_ids]
    if unlinked:
        warnings.append(f"unlinked_facts:{','.join(unlinked)}")


def resolve_roles(labels: list, n: int, warnings: list[str]) -> list[str]:
    """LLM 이 표시한 문장 역할을 문장 순서대로 편다. 번호가 맞지 않거나 빠진 문장은 detail 로 두고 경고한다."""
    labeled = {label.id: label.role for label in labels if 0 <= label.id < n}
    missing = [i for i in range(n) if i not in labeled]
    if missing:
        warnings.append(f"missing_sentence_roles:{','.join(map(str, missing))}")
    return [labeled.get(i, "detail") for i in range(n)]


def make_key_points(drafts: list, roles: list[str], norm: NormalizedScript, warnings: list[str]) -> list[KeyPoint]:
    """Key Point 초안 → KeyPoint. 문장 번호로 근거 원문·위치를, 근거 문장 역할로 중요도를 정한다."""
    n = len(norm.sentences)
    key_points = []
    for i, draft in enumerate(drafts, 1):
        ids = sorted({j for j in draft.sentence_ids if 0 <= j < n})
        bad = sorted(set(draft.sentence_ids) - set(ids))
        if bad:
            warnings.append(f"invalid_sentence_id:KP{i}:{','.join(map(str, bad))}")
        sents = [norm.sentences[j] for j in ids]
        key_points.append(KeyPoint(
            id=f"KP{i}", content=draft.content, importance=importance_from_roles([roles[j] for j in ids]),
            sentence_indices=ids, source_quote=" ".join(x.text for x in sents),
            source_span=(sents[0].start, sents[-1].end) if sents else None,
            key_terms=list(getattr(draft, "key_terms", [])), fact_ids=[],
        ))
    return key_points


def new_rubric(
    script_name: str, slide: SlideScript, norm: NormalizedScript, keywords: list[Keyword],
    roles: list[str], core_claim: str, key_points: list[KeyPoint], facts: list[CriticalFact],
    warnings: list[str], final_config: str = "", **review,
) -> EvaluationRubric:
    config_hash = llm_config_hash()
    rubric_key = f"{norm.content_hash}|{config_hash}|{final_config}|{RUBRIC_SCHEMA_VERSION}"
    return EvaluationRubric(
        meta=RubricMeta(
            rubric_id=hashlib.sha256(rubric_key.encode()).hexdigest()[:16],
            script_name=script_name,
            slide_number=slide.slide_number,
            content_hash=norm.content_hash,
            rubric_schema_version=RUBRIC_SCHEMA_VERSION,
            llm_model=MODEL,
            llm_config_hash=config_hash,
            final_config_hash=final_config,
            created_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
        ),
        normalized_script=norm.text,
        sentences=norm.sentences,
        sentence_roles=roles,
        core_claim=core_claim,
        key_points=key_points,
        critical_facts=facts,
        keywords=keywords,
        warnings=warnings,
        **review,
    )


def draft_rubric(
    script_name: str,
    slide: SlideScript,
    norm: NormalizedScript,
    facts: list[CriticalFact],
    keywords: list[Keyword],
    semantics: SlideSemanticAnalysis,
    deck_texts: list[str],
) -> EvaluationRubric:
    """1차 결과 정리·검증 (코드): 규칙 분석과 LLM 1차 분석을 합치고, 문제를 warnings 로 남긴다.

    이 초안과 경고가 6장 최종 결론(LLM)의 입력이 된다.
    deck_texts 는 같은 발표의 정규화된 슬라이드 대본들이다 (key_term 이 발표 공통 어휘인지 볼 때 쓴다).
    """
    warnings: list[str] = []
    roles = resolve_roles(semantics.sentence_roles, len(norm.sentences), warnings)
    key_points = make_key_points(semantics.key_points, roles, norm, warnings)
    facts = merge_key_terms(key_points, [f.model_copy(deep=True) for f in facts], norm, deck_texts, warnings)
    link_facts_to_key_points(key_points, facts, warnings)
    validate_rubric(key_points, facts, roles, warnings)
    return new_rubric(script_name, slide, norm, keywords, roles, semantics.core_claim, key_points, facts, warnings)

## 6. LLM API 최종 결론

1차 분석을 그대로 쓰지 않고, **LLM 을 한 번 더 불러 최종 판단**을 내립니다. 입력은 네 가지입니다 (`final_user_message`).

| 입력 | 내용 |
|---|---|
| 대본 | 문장 번호(`[S0]` …)가 붙은 정규화 대본 |
| 1차 분석 | 문장 역할, core_claim, Key Point (중요도 포함) |
| 규칙 기반 분석 | 수치 사실 (자동 채점이라 판단 대상 아님) + **이름·용어 후보** (규칙이 찾은 고유명사 + 1차 분석 key_terms, 등장 슬라이드 수·코드 판정 포함) |
| 코드 검증 경고 | 5장 `validate_rubric` 등이 찾은 문제 (claim 이 여러 개, 빠진 문장, 대본에 없는 수치 …) |

출력(`FinalReview`)은 최종 문장 역할, core_claim, Key Point, **이름·용어 후보별 유지/제외 결정과 이유**, 1차에서 **바꾼 점**입니다.
- 1차 분석을 기본으로 하고 경고가 가리키는 문제와 명백한 오류만 고치게 했습니다. 판단이 애매할 때마다 바꾸면 오히려 흔들림이 커지기 때문입니다.
- 이름·용어는 여기서 최종 결정합니다. 5장의 코드 판정은 참고 정보이고, 한 슬라이드에만 나오는 일반 명사처럼 코드가 못 거르는 것도 판단합니다.
- 수치 사실은 결정 대상이 아닙니다. 규칙이 찾은 수치는 항상 평가 기준에 들어갑니다.

`finalize_rubric` 이 최종 판단을 평가 기준으로 조립합니다. 중요도·사실 연결·검증은 5장과 같은 규칙으로 다시 계산하고,
1차 경고(`draft_warnings`)·바꾼 점(`review_changes`)·이름 결정(`name_decisions`)을 함께 저장해 무엇이 왜 바뀌었는지 남깁니다.

In [9]:
class FinalKeyPoint(BaseModel):
    content: str = Field(description="핵심 내용 한 문장. '~이다/~한다'로 끝나는 평서문. 대본의 수치·고유명사는 그대로 유지")
    sentence_ids: list[int] = Field(description="근거 문장 번호. 이어지는 문장 1개 이상")


class NameVerdict(BaseModel):
    candidate_id: str = Field(description="이름·용어 후보 번호 (N1, N2 …)")
    keep: bool = Field(description="발표자가 이 말을 그대로 해야 전달된 것으로 볼 수 있으면 true")
    reason: str = Field(description="짧은 이유")


class FinalReview(BaseModel):
    # 필드 순서가 생성 순서다: 문장 역할을 먼저 확정한 뒤 Key Point 와 이름·용어를 정한다
    sentence_roles: list[SentenceLabel] = Field(description="최종 문장 역할. 모든 문장에 하나씩")
    core_claim: str = Field(description="최종 핵심 주장 한 문장")
    key_points: list[FinalKeyPoint] = Field(description="최종 Key Point")
    name_verdicts: list[NameVerdict] = Field(description="이름·용어 후보마다 하나씩")
    changes: list[str] = Field(description="1차 분석에서 바꾼 점과 이유. 바꾼 게 없으면 빈 목록")


FINAL_SYSTEM_PROMPT = """
너는 발표 코칭 서비스의 채점 기준 검토자야.
입력은 슬라이드 한 장의 대본(문장 번호 포함), LLM 1차 분석 결과, 규칙 기반 분석 결과, 코드 검증 경고야.
이것들을 종합해서 채점 기준의 최종 결론을 내려. 결과는 발표자의 실제 발화(STT)와 비교해 핵심 내용이 전달됐는지 채점하는 데 쓰여.

# 원칙
- 1차 분석을 기본으로 해. 코드 검증 경고가 가리키는 문제와 대본과 명백히 어긋나는 부분만 고치고, 판단이 애매하면 1차 분석을 그대로 둬.
- 바꾼 점은 changes 에 "무엇을 왜 바꿨는지" 한 줄씩 적어. 바꾼 게 없으면 빈 목록으로 둬.

# sentence_roles
- 모든 문장에 하나씩 표시해.
  - claim: 이 슬라이드가 청중에게 남기려는 핵심 주장을 가장 직접적으로 말하는 문장. 슬라이드당 정확히 1문장.
    팀 소개·목차처럼 대등한 항목을 나열하기만 하는 슬라이드에는 두지 마.
  - evidence: claim 을 뒷받침하는 근거·기능·수치. 빠지면 claim 이 설득력을 잃는 문장.
  - detail: 예시·배경·부연 설명. 빠져도 claim 은 그대로 전달되는 문장.
  - skip: 인사, 전환 멘트, 마무리 인사처럼 전달 여부를 따질 필요가 없는 문장.
- 규칙 기반 분석에서 수치 사실이 나온 문장은 skip 으로 두지 마.

# core_claim
- claim 문장을 바탕으로 핵심 주장을 한 문장으로 정리해.

# key_points
- skip 이 아닌 모든 문장이 어느 key point 에든 들어가야 해. sentence_ids 에는 이어지는 문장만 묶어.
- key point 하나에는 주장·사실 하나만 담고, 대본 길이에 맞춰 2~6개로 만들어.
- content 는 '~이다', '~한다'로 끝나는 평서문 한 문장으로 써. 대본의 수치·고유명사는 그대로 쓰고, 대본에 없는 수치나 내용은 넣지 마.

# name_verdicts
- '이름·용어 후보' 전부에 대해 candidate_id 별로 하나씩 결정해.
- keep=true: 발표자가 이 말을 그대로 해야 전달된 것으로 볼 수 있는 이름·용어 (서비스·제품·팀·사람·기관·모델 이름, 이 분야 전문 용어).
- keep=false: 다른 말로 바꿔 말해도 되는 일반 명사, 4어절 이상의 구절이나 제목, 이름이 아닌데 이름으로 잡힌 말.
- '코드 판정' 은 참고용이야. 최종 판단은 네가 해.
"""

WARNING_TEXT = {
    "too_many_claims": "claim 문장이 여러 개",
    "uncovered_sentences": "어느 Key Point 에도 들어가지 않은 문장",
    "missing_sentence_roles": "역할이 빠진 문장",
    "invalid_sentence_id": "없는 문장 번호",
    "too_many_critical": "critical Key Point 가 여러 개",
    "no_critical_key_point": "critical Key Point 없음 (나열형 슬라이드면 정상)",
    "no_key_points": "Key Point 없음",
    "too_many_key_points": "Key Point 가 6개 초과",
    "key_point_without_sentence": "근거 문장이 없는 Key Point",
    "unsupported_number_in_key_point": "Key Point 내용에 대본에 없는 수치",
    "key_term_not_found": "대본에 없는 key_term",
    "key_term_not_fact": "사실로 올리지 않은 key_term",
    "unlinked_facts": "어느 Key Point 에도 연결되지 않은 사실",
}
REJECTION_TEXT = {"phrase": "4어절 이상의 구절", "common_in_deck": "이 발표의 여러 슬라이드에 나오는 일반 명사"}


def final_config_hash() -> str:
    payload = json.dumps({
        "model": MODEL, "prompt": FINAL_SYSTEM_PROMPT, "schema": FinalReview.model_json_schema(),
    }, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


def build_final_llm():
    """FinalReview 모양으로만 답하도록 고정한 LLM."""
    model = ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, model=MODEL, max_retries=2)
    return model.with_structured_output(FinalReview)


def name_candidates(draft: EvaluationRubric, rule_facts: list[CriticalFact], norm: NormalizedScript,
                    deck_texts: list[str]) -> list[CriticalFact]:
    """최종 결론이 판단할 이름·용어 후보: 규칙이 찾은 고유명사 + 1차 분석의 key_terms (걸러내지 않고 모두)."""
    key_points = [kp.model_copy(deep=True) for kp in draft.key_points]
    rule_copy = [f.model_copy(deep=True) for f in rule_facts]
    merged = merge_key_terms(key_points, rule_copy, norm, deck_texts, [], filter_terms=False)
    return [f for f in merged if f.type in NAME_TYPES]


def _describe_warning(warning: str) -> str:
    code, _, detail = warning.partition(":")
    return WARNING_TEXT.get(code, code) + (f" — {detail}" if detail else "")


def final_user_message(slide: SlideScript, norm: NormalizedScript, draft: EvaluationRubric,
                       rule_facts: list[CriticalFact], candidates: list[CriticalFact], deck_texts: list[str]) -> str:
    """최종 결론의 입력: 문장 번호가 붙은 대본 + 1차 분석 + 규칙 기반 분석 + 코드 검증 경고."""
    lines = [f"[슬라이드 {slide.slide_number}]", "## 대본"]
    lines += [SENTENCE_LINE.format(index=s.index, text=s.text) for s in norm.sentences]
    lines += ["", "## 1차 분석",
              "문장 역할: " + ", ".join(f"S{i}={role}" for i, role in enumerate(draft.sentence_roles)),
              f"core_claim: {draft.core_claim}", "Key Point:"]
    lines += [f"- {kp.id} [{kp.importance}] ({','.join(f'S{i}' for i in kp.sentence_indices)}) {kp.content}"
              for kp in draft.key_points]
    lines += ["", "## 규칙 기반 분석", "수치 사실 (코드가 자동으로 채점하므로 결정할 필요 없음):"]
    numeric = [f for f in rule_facts if f.type not in NAME_TYPES]
    lines += [f"- {f.value} → {f.normalized} (S{f.sentence_indices[0]})" for f in numeric] or ["- 없음"]
    lines.append("이름·용어 후보 (모두 결정 필요):")
    for i, fact in enumerate(candidates, 1):
        if fact.source == "llm":
            origin = "1차 분석 key_term"
        else:
            origin = "영문 이름" if re.search(r"[A-Za-z0-9]", fact.value) else "한글 고유명사 추정"
        n_slides = sum(bool(_term_spans(fact.value, text)) for text in deck_texts)
        hints = [origin, ",".join(f"S{j}" for j in sorted(set(fact.sentence_indices))), f"이 발표 슬라이드 {n_slides}개에 나옴"]
        verdict = key_term_rejection(fact.value, deck_texts)
        if verdict:
            hints.append("코드 판정: " + REJECTION_TEXT[verdict])
        lines.append(f"- N{i} '{fact.value}' ({' · '.join(hints)})")
    if not candidates:
        lines.append("- 없음")
    lines += ["", "## 코드 검증 경고"]
    lines += [f"- {_describe_warning(w)}" for w in draft.warnings] or ["- 없음"]
    return "\n".join(lines)


def review_final(message: str, final_llm) -> FinalReview:
    """LLM API (최종 결론): 1차 분석·규칙 분석·검증 경고를 보고 최종 채점 기준을 정한다."""
    return final_llm.invoke([("system", FINAL_SYSTEM_PROMPT), ("user", message)])


def finalize_rubric(
    script_name: str, slide: SlideScript, norm: NormalizedScript, rule_facts: list[CriticalFact],
    keywords: list[Keyword], draft: EvaluationRubric, review: FinalReview,
    candidates: list[CriticalFact], deck_texts: list[str],
) -> EvaluationRubric:
    """최종 결론 → Evaluation Rubric (코드). 중요도·사실 연결·검증은 1차와 같은 규칙으로 다시 계산한다.

    - 수치 사실은 규칙 결과를 그대로 쓴다 (LLM 이 빼지 못한다)
    - 이름·용어 후보는 최종 결론이 keep 으로 판단한 것만 사실로 올린다
    """
    warnings: list[str] = []
    roles = resolve_roles(review.sentence_roles, len(norm.sentences), warnings)
    key_points = make_key_points(review.key_points, roles, norm, warnings)

    verdicts = {v.candidate_id: v for v in review.name_verdicts}
    decisions, kept = [], []
    for i, cand in enumerate(candidates, 1):
        verdict = verdicts.get(f"N{i}")
        if verdict is None:  # 판단이 빠진 후보는 코드 기준으로 정하고 경고한다
            warnings.append(f"name_undecided:N{i}")
            keep, reason = key_term_rejection(cand.value, deck_texts) is None, "최종 결론이 판단하지 않아 코드 기준으로 결정"
        else:
            keep, reason = verdict.keep, verdict.reason
        decisions.append(NameDecision(value=cand.value, source=cand.source, keep=keep, reason=reason))
        if keep:
            kept.append(cand.model_copy(deep=True))

    facts = [f.model_copy(deep=True) for f in rule_facts if f.type not in NAME_TYPES] + kept
    facts.sort(key=lambda f: f.spans[0][0])
    for i, fact in enumerate(facts, 1):
        fact.id = f"CF{i}"
    link_facts_to_key_points(key_points, facts, warnings)
    for kp in key_points:  # Key Point 의 key_terms = 연결된 이름·용어 사실
        kp.key_terms = [f.value for f in facts if f.id in kp.fact_ids and f.type in NAME_TYPES]
    validate_rubric(key_points, facts, roles, warnings)

    return new_rubric(
        script_name, slide, norm, keywords, roles, review.core_claim, key_points, facts, warnings,
        final_config=final_config_hash(),
        draft_warnings=draft.warnings, review_changes=review.changes, name_decisions=decisions,
    )


print("final_config_hash:", final_config_hash())

final_config_hash: 83cb79b10a03


## 7. DB

로컬 SQLite 파일 `outputs/rubrics.sqlite` 를 씁니다. 노트북을 다시 실행해도 지우지 않습니다.

- `evaluation_rubrics` — 키는 `(script_name, slide_number)`. `script_name` 은 JSON 파일 이름입니다. STT 와 비교할 때 읽어 가는 최종 평가 기준.
  `meta.rubric_id` 는 (대본, 1차 분석 설정, 최종 결론 설정, 스키마)가 같으면 같은 값이라, 채점 결과에 함께 남기면 어떤 기준으로 채점했는지 추적할 수 있습니다
- `semantic_cache` — 1차 분석 결과. 키는 `(정규화 대본 해시, 1차 분석 설정 해시)`. **대본이 안 바뀐 슬라이드는 다시 부르지 않습니다**
- `final_cache` — 최종 결론 결과. 키는 `(입력 메시지 해시, 최종 결론 설정 해시)`. 대본·1차 분석·규칙 분석·경고가 모두 같으면 다시 부르지 않습니다
- `semantic_samples` — 12장 일관성 측정용 추가 1차 분석 응답
- 프롬프트·스키마·모델이 바뀌면 설정 해시가 달라져 다시 분석합니다

In [10]:
def connect_db(path: Path = DB_PATH) -> sqlite3.Connection:
    """로컬 SQLite DB. 실행을 반복해도 지우지 않으므로 LLM 캐시가 남는다."""
    path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(path)
    conn.executescript("""
        -- LLM 분기 결과 캐시. 정규화 대본과 LLM 설정이 같으면 다시 호출하지 않는다.
        CREATE TABLE IF NOT EXISTS semantic_cache (
            content_hash    TEXT NOT NULL,
            llm_config_hash TEXT NOT NULL,
            output_json     TEXT NOT NULL,
            created_at      TEXT NOT NULL,
            PRIMARY KEY (content_hash, llm_config_hash)
        );
        -- 일관성 측정용: 같은 입력으로 LLM 을 여러 번 부른 응답. 0번은 semantic_cache 의 응답을 쓴다
        CREATE TABLE IF NOT EXISTS semantic_samples (
            content_hash    TEXT    NOT NULL,
            llm_config_hash TEXT    NOT NULL,
            sample_index    INTEGER NOT NULL,
            output_json     TEXT    NOT NULL,
            created_at      TEXT    NOT NULL,
            PRIMARY KEY (content_hash, llm_config_hash, sample_index)
        );
        -- 최종 결론 캐시. 입력 메시지(대본 + 1차 분석 + 규칙 분석 + 경고)와 설정이 같으면 다시 호출하지 않는다
        CREATE TABLE IF NOT EXISTS final_cache (
            input_hash        TEXT NOT NULL,
            final_config_hash TEXT NOT NULL,
            output_json       TEXT NOT NULL,
            created_at        TEXT NOT NULL,
            PRIMARY KEY (input_hash, final_config_hash)
        );
        -- STT 와 비교할 때 읽어 갈 평가 기준
        CREATE TABLE IF NOT EXISTS evaluation_rubrics (
            script_name     TEXT    NOT NULL,
            slide_number    INTEGER NOT NULL,
            content_hash    TEXT    NOT NULL,
            rubric_json     TEXT    NOT NULL,
            created_at      TEXT    NOT NULL,
            PRIMARY KEY (script_name, slide_number)
        );
    """)
    return conn


def load_cached_semantics(conn: sqlite3.Connection, content_hash: str) -> SlideSemanticAnalysis | None:
    row = conn.execute(
        "SELECT output_json FROM semantic_cache WHERE content_hash = ? AND llm_config_hash = ?",
        (content_hash, llm_config_hash()),
    ).fetchone()
    return SlideSemanticAnalysis.model_validate_json(row[0]) if row else None


def save_cached_semantics(conn: sqlite3.Connection, content_hash: str, semantics: SlideSemanticAnalysis) -> None:
    conn.execute(
        "INSERT OR REPLACE INTO semantic_cache VALUES (?, ?, ?, ?)",
        (content_hash, llm_config_hash(), semantics.model_dump_json(), datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()


def _message_hash(message: str) -> str:
    return hashlib.sha256(message.encode("utf-8")).hexdigest()


def load_cached_final(conn: sqlite3.Connection, message: str) -> FinalReview | None:
    row = conn.execute(
        "SELECT output_json FROM final_cache WHERE input_hash = ? AND final_config_hash = ?",
        (_message_hash(message), final_config_hash()),
    ).fetchone()
    return FinalReview.model_validate_json(row[0]) if row else None


def save_cached_final(conn: sqlite3.Connection, message: str, review: FinalReview) -> None:
    conn.execute(
        "INSERT OR REPLACE INTO final_cache VALUES (?, ?, ?, ?)",
        (_message_hash(message), final_config_hash(), review.model_dump_json(), datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()


def save_rubric(conn: sqlite3.Connection, rubric: EvaluationRubric) -> None:
    m = rubric.meta
    conn.execute(
        "INSERT OR REPLACE INTO evaluation_rubrics VALUES (?, ?, ?, ?, ?)",
        (m.script_name, m.slide_number, m.content_hash, rubric.model_dump_json(), m.created_at),
    )
    conn.commit()


def load_rubric(conn: sqlite3.Connection, script_name: str, slide_number: int) -> EvaluationRubric | None:
    row = conn.execute(
        "SELECT rubric_json FROM evaluation_rubrics WHERE script_name = ? AND slide_number = ?",
        (script_name, slide_number),
    ).fetchone()
    return EvaluationRubric.model_validate_json(row[0]) if row else None

## 8. 파이프라인

`run_script_analysis` 한 번이 그림 전체입니다.
① 1차 분석 LLM 호출은 스레드 풀에서 병렬로 보내고, 그동안 메인 스레드가 규칙 기반 분석을 합니다 → ② 1차 결과 정리·검증(코드)
→ ③ 최종 결론 LLM 호출(병렬) → ④ 조립·저장(코드).
한 슬라이드의 LLM 호출이 실패해도 나머지 결과는 저장하고, 실패한 슬라이드는 `stats["failed"]` 로 알려 줍니다 (다시 돌리면 그 슬라이드만 호출).

In [11]:
EMPTY_SEMANTICS = SlideSemanticAnalysis(sentence_roles=[], core_claim="", key_points=[])


def _call_in_parallel(jobs: dict, fn, max_workers: int) -> tuple[dict, dict]:
    """jobs = {i: 인자 튜플} 을 스레드 풀에서 부른다. 성공 결과와 실패 메시지를 나눠 돌려준다."""
    done, errors = {}, {}
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {i: pool.submit(fn, *args) for i, args in jobs.items()}
        for i, future in futures.items():
            try:
                done[i] = future.result()
            except Exception as e:  # 한 장이 실패해도 나머지 (이미 비용을 낸) 결과는 살린다
                errors[i] = f"{type(e).__name__}: {e}"
    return done, errors


def run_script_analysis(
    path: Path,
    conn: sqlite3.Connection,
    llm,
    final_llm,
    max_workers: int = 4,
) -> tuple[list[EvaluationRubric], dict]:
    """Script Analysis Pipeline. 대본 등록·수정 시 한 번 실행한다.

    슬라이드마다 ① 규칙 기반 분석 1회 ∥ LLM 1차 분석 1회 → ② 1차 결과 정리·검증(코드)
    → ③ LLM 최종 결론 1회 → ④ 조립(코드)·저장.
    입력은 대본 JSON 파일 하나(= 발표 하나)이고, 결과는 파일 이름(확장자 제외)으로 저장한다.
    LLM 호출이 실패한 슬라이드는 Rubric 을 저장하지 않고 stats["failed"] 에 남긴다.
    성공한 호출은 캐시에 남으므로 다시 돌리면 실패한 슬라이드만 호출한다.
    """
    script_name = path.stem
    slides = load_script_json(path)
    norms = [normalize_script(s.script) for s in slides]
    deck_texts = [n.text for n in norms]
    keywords = extract_keywords_tfidf(norms)
    failed: dict[int, str] = {}

    # ① 규칙 기반 분석 ∥ LLM 1차 분석 (캐시에 없고 읽을 내용이 있는 슬라이드만 호출)
    semantics = {i: load_cached_semantics(conn, n.content_hash) for i, n in enumerate(norms)}
    analysis_jobs = {i: (slides[i], norms[i], llm) for i in semantics if semantics[i] is None and norms[i].text}
    with ThreadPoolExecutor(max_workers=1) as waiter:  # LLM 호출을 기다리는 동안 메인 스레드는 규칙 분석을 한다
        pending = waiter.submit(_call_in_parallel, analysis_jobs, analyze_semantics, max_workers)
        facts = [extract_critical_facts(n) for n in norms]
        analyzed, analysis_errors = pending.result()
    for i, sem in analyzed.items():
        semantics[i] = sem
        save_cached_semantics(conn, norms[i].content_hash, sem)  # sqlite 연결은 메인 스레드에서만
    failed.update({slides[i].slide_number: f"1차 분석 {msg}" for i, msg in analysis_errors.items()})

    # ② 1차 결과 정리·검증 (코드)
    ready = [i for i in range(len(slides)) if i not in analysis_errors]
    drafts = {i: draft_rubric(script_name, slides[i], norms[i], facts[i], keywords[i],
                              semantics[i] or EMPTY_SEMANTICS, deck_texts) for i in ready}

    # ③ LLM 최종 결론 (읽을 내용이 있는 슬라이드만)
    inputs = {}
    for i in ready:
        if norms[i].text:
            candidates = name_candidates(drafts[i], facts[i], norms[i], deck_texts)
            inputs[i] = (candidates, final_user_message(slides[i], norms[i], drafts[i], facts[i], candidates, deck_texts))
    reviews = {i: load_cached_final(conn, message) for i, (_, message) in inputs.items()}
    final_jobs = {i: (inputs[i][1], final_llm) for i in reviews if reviews[i] is None}
    reviewed, final_errors = _call_in_parallel(final_jobs, review_final, max_workers)
    for i, review in reviewed.items():
        reviews[i] = review
        save_cached_final(conn, inputs[i][1], review)
    failed.update({slides[i].slide_number: f"최종 결론 {msg}" for i, msg in final_errors.items()})

    # ④ 조립(코드)·저장
    rubrics = []
    for i in ready:
        if i in final_errors:
            continue
        if i not in inputs:  # 읽을 내용이 없는 슬라이드는 1차 결과가 곧 최종이다
            rubric = drafts[i]
        else:
            rubric = finalize_rubric(script_name, slides[i], norms[i], facts[i], keywords[i],
                                     drafts[i], reviews[i], inputs[i][0], deck_texts)
        save_rubric(conn, rubric)
        rubrics.append(rubric)

    stats = {
        "slides": len(slides),
        "analysis_calls": len(analysis_jobs),
        "final_calls": len(final_jobs),
        "failed": failed,
    }
    return rubrics, stats


def rubric_summary(rubrics: list[EvaluationRubric]) -> pd.DataFrame:
    rows = []
    for r in rubrics:
        importance = Counter(kp.importance for kp in r.key_points)
        rows.append({
            "slide": r.meta.slide_number,
            "KP": len(r.key_points),
            "critical/high/normal": f"{importance['critical']}/{importance['high']}/{importance['normal']}",
            "facts(rule/llm)": f"{sum(f.source == 'rule' for f in r.critical_facts)}/{sum(f.source == 'llm' for f in r.critical_facts)}",
            "1차 경고": len(r.draft_warnings),
            "최종 결론이 바꾼 점": len(r.review_changes),
            "최종 경고": "; ".join(r.warnings),
            "top keywords": ", ".join(k.term for k in r.keywords[:4]),
        })
    return pd.DataFrame(rows)

## 9. 실행

`analysis_calls` 는 1차 분석, `final_calls` 는 최종 결론 호출 수입니다. 캐시가 비어 있으면 20장 × 2회 = 40회 부르고, 다시 실행하면 0회입니다.

In [12]:
# DB 는 지우지 않는다. 같은 대본·같은 LLM 설정이면 캐시를 써서 API 를 다시 부르지 않는다.
# 처음부터 다시 분석하려면 outputs/rubrics.sqlite 를 지우고 실행한다.
if "conn" in globals():
    conn.close()
conn = connect_db()
llm = build_semantic_llm()        # 1차 분석
final_llm = build_final_llm()     # 최종 결론

all_rubrics: dict[str, list[EvaluationRubric]] = {}
failures: dict[str, dict[int, str]] = {}
for path in SCRIPT_FILES:
    rubrics, stats = run_script_analysis(path, conn, llm, final_llm)
    all_rubrics[path.stem] = rubrics
    failures[path.stem] = stats.pop("failed")
    print(path.name, stats, "실패한 슬라이드:", sorted(failures[path.stem]) or "-")

# 실패한 슬라이드가 있으면 여기서 멈춘다. 성공한 슬라이드는 캐시에 남으므로 원인을 고치고 다시 실행하면 실패한 것만 호출한다.
first_error = next((msg for failed in failures.values() for msg in failed.values()), None)
if first_error:
    raise RuntimeError(f"LLM 호출 실패 — 첫 오류: {first_error}")

가상대본1.json {'slides': 9, 'analysis_calls': 0, 'final_calls': 0} 실패한 슬라이드: -


가상대본2.json {'slides': 11, 'analysis_calls': 0, 'final_calls': 0} 실패한 슬라이드: -


### 슬라이드별 요약

In [13]:
pd.set_option("display.max_colwidth", 80)
for script_name, rubrics in all_rubrics.items():
    print(f"■ {script_name}")
    display(rubric_summary(rubrics))

■ 가상대본1


,slide,KP,critical/high/normal,facts(rule/llm),1차 경고,최종 결론이 바꾼 점,최종 경고,top keywords
0,1,4,1/2/1,3/1,0,0,,"빈자리, 하루, 하루 평균, 서비스"
1,2,5,1/3/1,1/0,0,0,,"현장, 예약, 현장 순번, 모바일"
2,3,5,1/3/1,4/0,0,0,,"기록, 자리, 핵심, 차례"
3,4,4,1/3/0,7/1,0,0,,"모델, 오차, 시험 기간, 시험"
4,5,6,1/3/2,2/0,0,0,,"화면, 열람실별, 한눈, 이용자 화면"
5,6,5,1/3/1,12/0,0,0,,"시범, 시범 운영, 운영, 비율"
6,7,3,1/1/1,7/0,1,1,,"대학 도서관, 대학, 초기 대상, 예약 시스템"
7,8,4,1/2/1,4/2,0,0,,"학습 통계, 통계, 추가, 추가 상품"
8,9,3,1/2/0,5/0,0,0,,"단계, 상반기, 카페, 스터디 카페"


■ 가상대본2


,slide,KP,critical/high/normal,facts(rule/llm),1차 경고,최종 결론이 바꾼 점,최종 경고,top keywords
0,1,5,1/2/2,2/0,1,1,,"동네 빵집, 동네, 빵집, 하루 생산량"
1,2,3,0/0/3,0/4,1,0,no_critical_key_point,"팀원, 화면 개발, 한도윤, 한도윤 팀원"
2,3,4,1/2/1,3/1,0,0,,"형식, 행사, 데이터, 판매"
3,4,3,1/2/0,3/1,0,0,,"기간, 판매량 규모, Prophet, 평가"
4,5,5,1/3/1,4/0,0,0,,"기존, 품목, 오차, 폐기율"
5,6,5,1/2/2,1/0,0,0,,"손해, 원가, 생산량, 주문"
6,7,2,1/1/0,0/1,2,1,,"추천, 추천 생산량, 생산량, 에이전트"
7,8,2,1/1/0,2/1,2,0,,"할인, 리포트, 에이전트, 할인 정보"
8,9,4,1/2/1,8/0,0,0,,"폐기, 요금, 비용, 효과"
9,10,4,1/1/2,2/0,1,0,,"생산량, 품목별 예측, 날짜, 모습"


### 슬라이드 상세 — Key Point 와 연결된 Critical Fact

In [14]:
def key_points_table(rubric: EvaluationRubric) -> pd.DataFrame:
    return pd.DataFrame([
        {"id": kp.id, "importance": kp.importance, "content": kp.content, "sentences": kp.sentence_indices,
         "key_terms": kp.key_terms, "facts": kp.fact_ids}
        for kp in rubric.key_points
    ])


def linked_facts_table(rubric: EvaluationRubric) -> pd.DataFrame:
    return pd.DataFrame([
        {"id": f.id, "type": f.type, "value": f.value, "normalized": f.normalized,
         "qualifier": f.qualifier, "source": f.source, "key_points": f.key_point_ids, "importance": f.importance}
        for f in rubric.critical_facts
    ])


def show_rubric(rubric: EvaluationRubric) -> None:
    m = rubric.meta
    print(f"[{m.script_name} / 슬라이드 {m.slide_number}]")
    print("문장 역할:", ", ".join(f"S{i} {role}" for i, role in enumerate(rubric.sentence_roles)))
    print("Core Claim:", rubric.core_claim)
    display(key_points_table(rubric))
    display(linked_facts_table(rubric))
    print("Keywords:", ", ".join(f"{k.term}({k.score:.2f})" for k in rubric.keywords))
    print("1차 검증 경고:", rubric.draft_warnings or "-")
    print("최종 결론이 바꾼 점:", rubric.review_changes or "-")
    print("이름·용어 결정:", [f"{d.value}={'유지' if d.keep else '제외'}({d.reason})" for d in rubric.name_decisions] or "-")
    print("최종 경고:", rubric.warnings or "-")


show_rubric(all_rubrics["가상대본1"][5])   # 수치가 많은 슬라이드
show_rubric(all_rubrics["가상대본2"][1])  # 사람 이름(LLM key_terms 보완)이 필요한 슬라이드

[가상대본1 / 슬라이드 6]
문장 역할: S0 detail, S1 claim, S2 evidence, S3 evidence, S4 evidence
Core Claim: 시범 운영 결과 빈자리를 찾는 데 걸린 시간이 평균 23분에서 13분으로 약 42퍼센트 줄었다.


,id,importance,content,sentences,key_terms,facts
0,KP1,normal,2025년 3월부터 8주 동안 제휴 도서관 3곳에서 시범 운영을 했다.,[0],[],"[CF1, CF2, CF3]"
1,KP2,critical,빈자리를 찾는 데 걸린 시간이 평균 23분에서 13분으로 약 42퍼센트 줄었다.,[1],[],"[CF4, CF5, CF6, CF7]"
2,KP3,high,알림을 받고 10분 안에 자리를 잡은 비율이 76%이다.,[2],[],"[CF8, CF9]"
3,KP4,high,노쇼 비율이 18%에서 7%로 낮아졌다.,[3],[],"[CF10, CF11]"
4,KP5,high,시범 운영 뒤 설문에서 이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답했다.,[4],[],[CF12]


,id,type,value,normalized,qualifier,source,key_points,importance
0,CF1,date,2025년 3월,2025-03,NaN,rule,[KP1],normal
1,CF2,duration,8주 동안,8주,NaN,rule,[KP1],normal
2,CF3,quantity,3곳,3곳,NaN,rule,[KP1],normal
3,CF4,quantity,"1,860명","1,860명",NaN,rule,[KP2],critical
4,CF5,duration,평균 23분,23분,평균,rule,[KP2],critical
5,CF6,duration,13분,13분,NaN,rule,[KP2],critical
6,CF7,percentage,약 42퍼센트,42%,약,rule,[KP2],critical
7,CF8,duration,10분,10분,NaN,rule,[KP3],high
8,CF9,percentage,76%,76%,NaN,rule,[KP3],high
9,CF10,percentage,18%,18%,NaN,rule,[KP4],high


Keywords: 시범(0.38), 시범 운영(0.38), 운영(0.38), 비율(0.32), 퍼센트(0.22), 동안(0.22), 동안 제휴(0.22), 설문(0.22)
1차 검증 경고: -
최종 결론이 바꾼 점: -
이름·용어 결정: -
최종 경고: -
[가상대본2 / 슬라이드 2]
문장 역할: S0 skip, S1 detail, S2 detail, S3 detail
Core Claim: 


,id,importance,content,sentences,key_terms,facts
0,KP1,normal,한도윤 팀원은 판매량 예측 모델과 성능 검증을 맡은 팀원이다.,[1],[한도윤],[CF1]
1,KP2,normal,서민재 팀원은 서비스 기획과 대화형 에이전트를 만든 팀원이다.,[2],"[서민재, 대화형 에이전트]","[CF2, CF3]"
2,KP3,normal,"오예린 팀원은 데이터 수집과 화면 개발, 발표 자료 정리를 담당한 팀원이다.",[3],[오예린],[CF4]


,id,type,value,normalized,qualifier,source,key_points,importance
0,CF1,term,한도윤,한도윤,None,llm,[KP1],normal
1,CF2,term,서민재,서민재,None,llm,[KP2],normal
2,CF3,term,대화형 에이전트,대화형 에이전트,None,llm,[KP2],normal
3,CF4,term,오예린,오예린,None,llm,[KP3],normal


Keywords: 팀원(0.38), 화면 개발(0.18), 한도윤(0.18), 한도윤 팀원(0.18), 정리(0.18), 팀원별(0.18), 팀원별 역할(0.18), 서민재 팀원(0.18)
1차 검증 경고: ['no_critical_key_point']
최종 결론이 바꾼 점: -
이름·용어 결정: ['한도윤=유지(팀원 이름이므로 그대로 전달되어야 한다.)', '서민재=유지(팀원 이름이므로 그대로 전달되어야 한다.)', '대화형 에이전트=유지(서비스의 전문 기능 용어이므로 그대로 전달되어야 한다.)', '오예린=유지(팀원 이름이므로 그대로 전달되어야 한다.)']
최종 경고: ['no_critical_key_point']


### 저장된 Evaluation Rubric (JSON)

In [15]:
# DB 에 저장된 Evaluation Rubric 원본 (STT 와 비교할 때 읽어 갈 모양)
stored = load_rubric(conn, "가상대본1", 7)
print(json.dumps(stored.model_dump(exclude={"sentences", "normalized_script", "keywords"}), ensure_ascii=False, indent=2))

{
  "meta": {
    "rubric_id": "94130ca6731b9a47",
    "script_name": "가상대본1",
    "slide_number": 7,
    "content_hash": "2c80d08e9afdcd4a1ad597ac61a127130a7532b1085ce5a0a6aa508ce684211d",
    "rubric_schema_version": "1.0",
    "llm_model": "openai/gpt-5.6-luna",
    "llm_config_hash": "f7b2ef94d4e7",
    "final_config_hash": "83cb79b10a03",
    "created_at": "2026-09-25T04:54:27+00:00"
  },
  "sentence_roles": [
    "evidence",
    "claim",
    "detail"
  ],
  "core_claim": "열람실 좌석 예약 시스템 시장의 초기 대상은 약 900곳이며 연간 시장 규모는 약 24억 원에서 36억 원이다.",
  "key_points": [
    {
      "id": "KP1",
      "content": "국내 공공도서관은 약 1,200곳이고 대학 도서관은 약 430곳이다.",
      "importance": "high",
      "sentence_indices": [
        0
      ],
      "source_quote": "국내 공공도서관은 약 1,200곳이고, 대학 도서관은 약 430곳입니다.",
      "source_span": [
        0,
        40
      ],
      "key_terms": [],
      "fact_ids": [
        "CF1",
        "CF2"
      ]
    },
    {
      "id": "KP2",
      "content": "열람실 좌석 예약 시스템을 이미 쓰는 초기 대

## 10. 대본 수정 시 재분석

이 파이프라인은 대본이 등록·수정될 때 한 번 돕니다. 한 문장만 고친 JSON(`가상대본2_수정.json`)을 넣으면 **바뀐 슬라이드만** 1차 분석·최종 결론을 다시 부르고 나머지는 캐시를 씁니다.
원본과 수정본 Rubric 은 파일 이름별로 따로 저장되어 서로 비교할 수 있습니다. LLM 응답은 호출마다 조금씩 달라서, 고친 슬라이드는 고치지 않은 문장의 Key Point 표현도 바뀔 수 있습니다.

In [16]:
# 가상 시나리오: 발표자가 가상대본2 슬라이드 5의 결과 문장 하나만 고친 JSON 을 다시 넣었다
V2_OLD = "사장님이 감으로 정하던 기존 방식과 비교하면 폐기율이 약 18퍼센트 줄었습니다."
V2_NEW = "사장님이 감으로 정하던 기존 방식과 비교하면 폐기율이 약 23퍼센트 줄었고, 오후 품절 알림의 정확도는 91퍼센트였습니다."

edited = json.loads((SCRIPT_DIR / "가상대본2.json").read_text(encoding="utf-8"))
slide5 = next(item for item in edited if item["slide_number"] == 5)
assert V2_OLD in slide5["script"]
slide5["script"] = slide5["script"].replace(V2_OLD, V2_NEW)

EDITED_PATH = ROOT / "outputs" / "가상대본2_수정.json"
EDITED_PATH.write_text(json.dumps(edited, ensure_ascii=False, indent=2), encoding="utf-8")

rubrics_v2, stats_v2 = run_script_analysis(EDITED_PATH, conn, llm, final_llm)
print(f"{EDITED_PATH.name}:", stats_v2)  # 바뀐 슬라이드 하나만 1차 분석·최종 결론을 다시 부른다


def fact_values(rubric: EvaluationRubric) -> set[str]:
    return {f"{f.type}:{f.normalized}" for f in rubric.critical_facts}


before = load_rubric(conn, "가상대본2", 5)
after = load_rubric(conn, EDITED_PATH.stem, 5)
print("사라진 사실:", fact_values(before) - fact_values(after))
print("새로 생긴 사실:", fact_values(after) - fact_values(before))
display(key_points_table(after))

가상대본2_수정.json: {'slides': 11, 'analysis_calls': 0, 'final_calls': 0, 'failed': {}}
사라진 사실: {'percentage:18%'}
새로 생긴 사실: {'percentage:91%', 'percentage:23%'}


,id,importance,content,sentences,key_terms,facts
0,KP1,critical,기존 방식과 비교해 폐기율은 약 23퍼센트 줄었고 오후 품절 알림의 정확도는 91퍼센트이다.,[1],[],"[CF1, CF2]"
1,KP2,high,품목별 판매량 예측 오차는 MAPE 기준 11.4%이다.,[2],[MAPE],"[CF3, CF4]"
2,KP3,normal,새로 나온 품목은 판매 기록이 적어 예측 오차가 30% 이상으로 커진다.,[3],[],[CF5]
3,KP4,high,예측 오차가 큰 새 품목은 비슷한 기존 품목의 판매 흐름을 빌려 와 예측하도록 보완한다.,[4],[],[]
4,KP5,high,예측이 불안정한 품목은 화면에 따로 표시해 사장님이 직접 판단할 수 있게 한다.,[5],[],[]


## 11. 점검

### 규칙 분기 — 결정적이므로 assert

In [17]:
def facts_of(script_name: str, slide_number: int) -> dict[str, CriticalFact]:
    rubric = load_rubric(conn, script_name, slide_number)
    return {f.normalized: f for f in rubric.critical_facts}


# 규칙 분기는 결정적이라 assert 로 확인한다
# 1) 정규화: 공백·따옴표만 정리하고 발화 표현은 그대로 둔다
assert normalize_script("‘재고 예보’입니다.\n\n  30퍼센트 남았다면").text == "'재고 예보'입니다. 30퍼센트 남았다면"

# 2) Critical Fact Parser: 가상 데이터
f4 = facts_of("가상대본1", 4)
assert f4["12,400,000건"].qualifier == "약" and "3:1" in f4 and "xgboost" in f4
f6 = facts_of("가상대본1", 6)
assert f6["2025-03"].type == "date" and "8주" in f6
assert f6["42%"].qualifier == "약" and f6["85%"].qualifier == "이상"
f7 = facts_of("가상대본1", 7)
assert {"2,400,000,000원", "3,600,000,000원", "1,200곳", "430곳", "900곳"} <= f7.keys()
f9 = facts_of("가상대본1", 9)
assert "2026-H1" in f9 and not any(f.value.endswith("단계") for f in f9.values())  # 서수 제외
assert {"15%", "20%"} <= facts_of("가상대본2", 1).keys()   # 15~20% 범위
assert {"18:00", "09:00"} <= facts_of("가상대본2", 8).keys()  # 오후 6시, 오전 9시
g9 = facts_of("가상대본2", 9)
assert "19,000원" in g9 and "6배" in g9                      # 1만 9천 원, 여섯 배

# 3) Critical Fact Parser: 실제 대본에 나올 법한 표기
PARSER_CASES = [
    ("시드 투자로 2억 3천만 원을 유치했습니다.", {"money:230,000,000원"}),
    ("점유율이 5%p 상승했습니다.", {"percentage:5%p"}),
    ("응답 속도를 10~20% 개선했습니다.", {"percentage:10%", "percentage:20%"}),
    ("5~10억 원 규모입니다.", {"money:500,000,000원", "money:1,000,000,000원"}),
    ("주요 고객은 20대 30대 직장인입니다.", {"quantity:20대", "quantity:30대"}),
    ("2025년 3분기에 출시하고 2026학년도부터 도입합니다.", {"date:2025-Q3", "date:2026"}),
    ("오후 3시 30분에 시작합니다.", {"time:15:30"}),
    ("매출이 두 배 늘었고 세 가지 기능을 한 달 만에 만들었습니다.", {"quantity:2배", "quantity:3가지", "duration:1개월"}),
    ("사용자는 삼십칠 퍼센트 증가했습니다.", {"percentage:37%"}),  # STT 식 한글 수사
    ("멘토와 멘티를 일대일로 연결합니다.", {"ratio:1:1"}),
    ("용량은 5,000 mAh 입니다.", {"quantity:5,000mAh"}),
    ("한 번 확인한 내용을 이대로 다시 보여 줍니다.", set()),  # 수사처럼 보이는 일반 표현
    # 단위 글자로 시작하는 일반 단어는 Kiwi 형태소 확인으로 걸러진다
    ("3원칙을 지키고 3프로젝트를 진행했습니다.", {"number:3"}),
    ("3시간 동안 5시리즈를 봤습니다.", {"duration:3시간", "number:5"}),
    ("두 배터리를 쓰고 한 대학에서 도입했습니다.", set()),
    ("둘 사이 명확한 차이가 있습니다.", set()),
]
for text, expected in PARSER_CASES:
    got = {f"{f.type}:{f.normalized}" for f in extract_critical_facts(normalize_script(text)) if f.type not in ("proper_noun", "term")}
    assert got == expected, (text, got)

# 4) 대본 수정 시: 바뀐 슬라이드만 다시 분석하고, 바뀐 수치가 사실에 반영된다 (이 셀만 다시 돌리면 호출 0)
assert stats_v2["slides"] == 11 and stats_v2["analysis_calls"] <= 1 and stats_v2["final_calls"] <= 1 and not stats_v2["failed"]
assert "23%" in facts_of(EDITED_PATH.stem, 5)
print("규칙 점검 통과")

규칙 점검 통과


### LLM 결과 품질 (최종 기준) — LLM 응답에 따라 달라지므로 지표로만 봅니다

In [18]:
def llm_quality_report(rubrics: list[EvaluationRubric]) -> pd.DataFrame:
    """LLM 분기 출력이 채점 기준으로 쓸 만한지 보는 지표. LLM 응답이 바뀌면 값도 바뀐다."""
    kps = [kp for r in rubrics for kp in r.key_points]
    facts = [f for r in rubrics for f in r.critical_facts]
    numeric = [f for f in facts if f.type not in ("proper_noun", "term")]
    importance = Counter(kp.importance for kp in kps)
    rows = {
        "Key Point 수": len(kps),
        "중요도 분포 (critical/high/normal)": f"{importance['critical']}/{importance['high']}/{importance['normal']}",
        "claim 문장이 2개 이상인 슬라이드": sum(w.startswith("too_many_claims") for r in rubrics for w in r.warnings),
        "Key Point 에 안 들어간 문장 (skip 제외)": sum(
            len(w.split(":", 1)[1].split(",")) for r in rubrics for w in r.warnings if w.startswith("uncovered_sentences")),
        "잘못된 문장 번호 / 역할이 빠진 문장": sum(
            w.startswith(("invalid_sentence_id", "missing_sentence_roles")) for r in rubrics for w in r.warnings),
        "critical 없는 슬라이드": sum(all(kp.importance != "critical" for kp in r.key_points) for r in rubrics),
        "Key Point 에 연결된 수치 사실": f"{sum(bool(f.key_point_ids) for f in numeric)}/{len(numeric)}",
        "LLM 이 보탠 용어(term) 사실": sum(f.source == "llm" for f in facts),
        "대본에 없는 key_term": sum(w.startswith("key_term_not_found") for r in rubrics for w in r.warnings),
        "이름·용어 후보 유지/제외 (최종 결론)": "{}/{}".format(
            sum(d.keep for r in rubrics for d in r.name_decisions), sum(not d.keep for r in rubrics for d in r.name_decisions)),
        "1차 검증 경고 수 → 최종 경고 수": "{} → {}".format(
            sum(len(r.draft_warnings) for r in rubrics), sum(len(r.warnings) for r in rubrics)),
        "최종 결론이 1차에서 바꾼 항목": sum(len(r.review_changes) for r in rubrics),
        "critical 이 2개 이상인 슬라이드": sum(w.startswith("too_many_critical") for r in rubrics for w in r.warnings),
        "대본에 없는 수치를 말한 Key Point": sum(w.startswith("unsupported_number") for r in rubrics for w in r.warnings),
    }
    return pd.DataFrame(rows.items(), columns=["지표", "값"])


display(llm_quality_report([r for rs in all_rubrics.values() for r in rs]))

# 사람 이름처럼 Kiwi 가 놓친 고유명사를 LLM key_terms 가 채웠는지
print("팀 소개 슬라이드의 이름 사실:", [
    f"{f.value}({f.source})" for f in load_rubric(conn, "가상대본2", 2).critical_facts
    if f.value in ("한도윤", "서민재", "오예린")
])

,지표,값
0,Key Point 수,80
1,중요도 분포 (critical/high/normal),19/40/21
2,claim 문장이 2개 이상인 슬라이드,0
3,Key Point 에 안 들어간 문장 (skip 제외),0
4,잘못된 문장 번호 / 역할이 빠진 문장,0
5,critical 없는 슬라이드,1
6,Key Point 에 연결된 수치 사실,57/57
7,LLM 이 보탠 용어(term) 사실,12
8,대본에 없는 key_term,0
9,이름·용어 후보 유지/제외 (최종 결론),25/6


팀 소개 슬라이드의 이름 사실: ['한도윤(llm)', '서민재(llm)', '오예린(llm)']


## 12. LLM 일관성 측정

채점 기준은 매번 같아야 합니다. 같은 대본으로 채점 기준을 `N_SAMPLES` 벌 만들어, **1차 결과와 최종 결과가 각각 얼마나 흔들리는지** 비교합니다.
k 번째 벌 = k 번째 1차 분석 응답 → 1차 결과(코드) → 최종 결론 응답 → 최종 결과(코드). 없는 응답만 새로 호출해 DB 에 저장합니다 (다시 실행하면 호출하지 않습니다).
Key Point 는 근거 문장 집합이 절반 이상 겹치면 같은 것으로 봅니다.

| 지표 | 뜻 |
|---|---|
| 문장 역할 일치율 | 같은 문장에 같은 역할(claim / evidence / detail / skip)을 준 비율 |
| KP 대응률 | 0번 기준의 Key Point 가 다른 기준에도 (근거가 겹치는 형태로) 있는 비율 |
| 중요도 일치율 | 대응된 Key Point 끼리 중요도가 같은 비율 |
| critical 위치 일치 | critical 로 고른 Key Point 의 근거가 같은 비율 (둘 다 critical 이 없으면 일치) |
| 이름·용어 일치 | 평가 기준에 들어간 이름·용어 사실 집합의 자카드 유사도 |
| 점수 차이 | 같은 가상 발화(문장마다 80% 확률로 말함)를 각 기준으로 채점했을 때 점수 차이 (100점 만점, `score_spread`). 기준이 흔들릴 때 점수가 얼마나 달라지는지 가늠하는 값입니다 |

In [19]:
# 슬라이드마다 채점 기준을 몇 벌 만들어 비교할지. 1 이면 측정하지 않는다
# (다른 노트북이 이 노트북을 %run 으로 불러올 때 RUBRIC_CONSISTENCY_SAMPLES=1 로 꺼서 추가 호출을 막는다)
N_SAMPLES = int(os.getenv("RUBRIC_CONSISTENCY_SAMPLES", "3"))


def load_samples(conn: sqlite3.Connection, content_hash: str) -> dict[int, SlideSemanticAnalysis]:
    rows = conn.execute(
        "SELECT sample_index, output_json FROM semantic_samples WHERE content_hash = ? AND llm_config_hash = ?",
        (content_hash, llm_config_hash()),
    ).fetchall()
    return {i: SlideSemanticAnalysis.model_validate_json(js) for i, js in rows}


def save_sample(conn: sqlite3.Connection, content_hash: str, index: int, semantics: SlideSemanticAnalysis) -> None:
    conn.execute(
        "INSERT OR REPLACE INTO semantic_samples VALUES (?, ?, ?, ?, ?)",
        (content_hash, llm_config_hash(), index, semantics.model_dump_json(), datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()


def sample_rubrics(path: Path, conn: sqlite3.Connection, llm, final_llm, n_samples: int = N_SAMPLES):
    """슬라이드마다 1차 결과와 최종 결과를 n_samples 벌씩 만든다.

    k 번째 벌 = k 번째 1차 분석 응답 → 1차 결과(코드) → 최종 결론 응답 → 최종 결과(코드).
    0번 1차 분석은 파이프라인이 캐시한 응답이다. 없는 LLM 응답만 새로 호출하고 DB 에 저장하므로 다시 실행하면 호출하지 않는다.
    """
    slides = load_script_json(path)
    norms = [normalize_script(s.script) for s in slides]
    deck_texts = [n.text for n in norms]
    keywords = extract_keywords_tfidf(norms)
    facts = [extract_critical_facts(n) for n in norms]

    samples = []
    for norm in norms:
        got = load_samples(conn, norm.content_hash)
        first = load_cached_semantics(conn, norm.content_hash)
        if first is not None:
            got[0] = first
        samples.append(got)
    jobs = {(i, k): (slides[i], norms[i], llm) for i, got in enumerate(samples) for k in range(n_samples) if k not in got}
    done, errors = _call_in_parallel(jobs, analyze_semantics, 4)
    if errors:
        raise RuntimeError(f"1차 분석 호출 실패: {next(iter(errors.values()))}")
    for (i, k), sem in done.items():
        samples[i][k] = sem
        save_sample(conn, norms[i].content_hash, k, sem)
    n_calls = len(jobs)

    drafts, inputs = {}, {}
    for i in range(len(slides)):
        for k in range(n_samples):
            draft = draft_rubric(path.stem, slides[i], norms[i], facts[i], keywords[i], samples[i][k], deck_texts)
            candidates = name_candidates(draft, facts[i], norms[i], deck_texts)
            drafts[i, k] = draft
            inputs[i, k] = (candidates, final_user_message(slides[i], norms[i], draft, facts[i], candidates, deck_texts))
    reviews = {key: load_cached_final(conn, message) for key, (_, message) in inputs.items()}
    jobs = {key: (inputs[key][1], final_llm) for key in reviews if reviews[key] is None}
    done, errors = _call_in_parallel(jobs, review_final, 4)
    if errors:
        raise RuntimeError(f"최종 결론 호출 실패: {next(iter(errors.values()))}")
    for key, review in done.items():
        reviews[key] = review
        save_cached_final(conn, inputs[key][1], review)
    n_calls += len(jobs)

    finals = {
        (i, k): finalize_rubric(path.stem, slides[i], norms[i], facts[i], keywords[i],
                                drafts[i, k], reviews[i, k], inputs[i, k][0], deck_texts)
        for (i, k) in drafts
    }
    return slides, norms, drafts, finals, n_calls


def _jaccard(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if a | b else 1.0


def rubric_view(rubric: EvaluationRubric):
    """비교에 쓰는 요약: 문장 역할, (근거 문장 집합, 중요도) 목록, 이름·용어 사실 집합."""
    return (
        rubric.sentence_roles,
        [(set(kp.sentence_indices), kp.importance) for kp in rubric.key_points],
        {f.normalized for f in rubric.critical_facts if f.type in NAME_TYPES},
    )


def compare_views(views: list) -> dict:
    """0번을 기준으로 나머지와 비교한다. Key Point 는 근거 문장 집합이 절반 이상 겹치면 같은 것으로 본다."""
    base_roles, base_kps, base_names = views[0]
    pairs = matched = agreed = critical_same = role_same = role_total = 0
    for roles, kps, _ in views[1:]:
        for ids, importance in base_kps:
            pairs += 1
            best = max(kps, key=lambda kp: _jaccard(ids, kp[0]), default=(set(), None))
            if ids and _jaccard(ids, best[0]) >= 0.5:
                matched += 1
                agreed += best[1] == importance
        base_crit = [ids for ids, imp in base_kps if imp == "critical"]
        other_crit = [ids for ids, imp in kps if imp == "critical"]
        critical_same += (not base_crit and not other_crit) or any(_jaccard(a, b) >= 0.5 for a in base_crit for b in other_crit)
        role_total += len(base_roles)
        role_same += sum(a == b for a, b in zip(base_roles, roles))
    return {
        "KP 수": "/".join(str(len(kps)) for _, kps, _ in views),
        "critical 수": "/".join(str(sum(imp == "critical" for _, imp in kps)) for _, kps, _ in views),
        "문장 역할 일치율": role_same / role_total if role_total else 1.0,
        "KP 대응률": matched / pairs if pairs else 1.0,
        "중요도 일치율": agreed / matched if matched else 1.0,
        "critical 위치 일치": critical_same / (len(views) - 1),
        "이름·용어 일치": sum(_jaccard(base_names, names) for _, _, names in views[1:]) / (len(views) - 1),
    }


def score_spread(views: list, n_sentences: int, trials: int = 300, seed: int = 0) -> list[float]:
    """같은 가상 발화를 각 기준으로 채점했을 때 점수 차이(최대 - 최소, 100점 만점).

    문장마다 80% 확률로 말했다고 가정하고, Key Point 는 근거 문장을 다 말하면 1, 일부면 0.5, 하나도 안 말하면 0 으로
    중요도 가중 평균을 낸다. 실제 채점 방식이 아니라 기준이 흔들릴 때 점수가 얼마나 달라지는지 가늠하는 용도다.
    """
    rng = random.Random(seed)
    spreads = []
    for _ in range(trials):
        said = [rng.random() < 0.8 for _ in range(n_sentences)]
        scores = []
        for _, kps, _ in views:
            num = den = 0.0
            for ids, importance in kps:
                if not ids:
                    continue
                ratio = sum(said[j] for j in ids) / len(ids)
                num += (1.0 if ratio == 1 else 0.0 if ratio == 0 else 0.5) * SCORE_WEIGHT[importance]
                den += SCORE_WEIGHT[importance]
            scores.append(100 * num / den if den else 0.0)
        spreads.append(max(scores) - min(scores))
    return spreads


if N_SAMPLES < 2:
    print("N_SAMPLES 가 1 이라 일관성 측정을 건너뜁니다.")
else:
    rows, spreads = [], {"1차": [], "최종": []}
    for path in SCRIPT_FILES:
        slides, norms, drafts, finals, n_calls = sample_rubrics(path, conn, llm, final_llm)
        print(f"{path.name}: 추가 호출 {n_calls}회")
        for i, slide in enumerate(slides):
            for stage, rubrics_by_key in (("1차", drafts), ("최종", finals)):
                views = [rubric_view(rubrics_by_key[i, k]) for k in range(N_SAMPLES)]
                spread = score_spread(views, len(norms[i].sentences))
                spreads[stage] += spread
                rows.append({"file": path.stem, "slide": slide.slide_number, "단계": stage,
                             **compare_views(views), "점수 차이 평균": sum(spread) / len(spread)})

    consistency_table = pd.DataFrame(rows)
    display(consistency_table[consistency_table["단계"] == "최종"].drop(columns="단계").round(2))
    rate_cols = ["문장 역할 일치율", "KP 대응률", "중요도 일치율", "critical 위치 일치", "이름·용어 일치"]
    summary = consistency_table.groupby("단계")[rate_cols].mean()
    for stage, values in spreads.items():
        values = sorted(values)
        summary.loc[stage, "점수 차이 평균"] = sum(values) / len(values)
        summary.loc[stage, "점수 차이 중앙값"] = values[len(values) // 2]
        summary.loc[stage, "점수 차이 상위 10%"] = values[int(0.9 * (len(values) - 1))]
    display(summary.round(2))

가상대본1.json: 추가 호출 0회


가상대본2.json: 추가 호출 0회


,file,slide,KP 수,critical 수,문장 역할 일치율,KP 대응률,중요도 일치율,critical 위치 일치,이름·용어 일치,점수 차이 평균
1,가상대본1,1,4/4/5,1/1/1,0.71,1.0,0.38,0.0,1.0,8.72
3,가상대본1,2,5/4/4,1/1/1,1.00,1.0,1.00,1.0,0.5,4.18
5,가상대본1,3,5/5/5,1/1/1,0.60,1.0,0.60,1.0,0.5,5.77
7,가상대본1,4,4/5/4,1/1/1,1.00,1.0,1.00,1.0,1.0,3.72
9,가상대본1,5,6/5/5,1/1/1,1.00,1.0,1.00,1.0,1.0,2.56
11,가상대본1,6,5/5/5,1/1/1,0.80,1.0,0.80,0.5,1.0,3.43
13,가상대본1,7,3/3/4,1/1/2,1.00,1.0,1.00,1.0,1.0,4.96
15,가상대본1,8,4/4/3,1/1/1,1.00,1.0,0.88,1.0,0.0,3.94
17,가상대본1,9,3/3/3,1/1/1,0.67,1.0,0.33,0.0,1.0,7.74
19,가상대본2,1,5/5/5,1/1/1,0.83,1.0,0.80,1.0,1.0,2.95


,문장 역할 일치율,KP 대응률,중요도 일치율,critical 위치 일치,이름·용어 일치,점수 차이 평균,점수 차이 중앙값,점수 차이 상위 10%
단계,,,,,,,,
1차,0.83,1.0,0.81,0.80,0.92,4.15,0.0,12.5
최종,0.84,1.0,0.81,0.82,0.77,4.27,0.0,12.5
